# Komputasi Hisab Prediksi Awal Ramadan — Reproduksi Fase 1 & Fase 2 (Google Colab)

**Judul Skripsi:** Komputasi Hisab Prediksi Awal Ramadan Berbasis Data Ephemeris NASA JPL Horizons
Menggunakan Algoritma Newton-Raphson — Muhammad Reinaldy Santoso Alaratte, 202210715004,
Universitas Bhayangkara Jakarta Raya, 2026.

Notebook ini adalah **porting 1:1** (konstanta dan logika tidak diubah) dari kode produksi
TypeScript aplikasi web sistem ini, disusun ulang di Google Colab agar seluruh alur, rumus, dan
hasil dapat dijalankan langsung, diperiksa baris-per-baris, dan dilampirkan sebagai bukti kodingan
pada sidang akhir.

Notebook mereproduksi **dua fase evaluasi** sesuai alur aplikasi:

- **Fase 1 — Evaluasi Konjungsi** (`/evaluasi-konjungsi`, `/api/konjungsi-periode`):
  memindai seluruh peristiwa konjungsi (ijtimak) geosentris pada rentang tahun yang ditentukan
  (default **2017–2026**, sesuai Bab IV skripsi), lalu mengklasifikasikan konjungsi mana yang
  menjadi **kandidat awal Ramadan** untuk tiap tahun Masehi berdasarkan siklus kalender Hijriah.
- **Fase 2 — Evaluasi History Global dan Local** (`/evaluasi`, `/api/evaluate`):
  untuk rentang tahun dan lokasi pengamatan yang sama (default **Kota Bekasi**), membandingkan
  tiga sumber tanggal 1 Ramadan secara berdampingan:
  - **Global** — skenario geosentris KHGT (grid saksi dunia; hanya sebagai info pembanding)
  - **Local** — Rule A / Rule B (Wujudul Hilal) pada lokasi pengamatan (evaluasi utama skripsi)
  - **Historis/Resmi** — pengumuman Sidang Isbat Kementerian Agama RI (pembanding historis terbatas)


## Catatan Penting Sebelum Menjalankan

1. **Live-only, tanpa data simulasi.** Notebook ini query LANGSUNG ke NASA/JPL Horizons API
   (`https://ssd.jpl.nasa.gov/api/horizons.api`). Sesuai kebijakan data akademik sistem asli,
   **tidak ada mode mock/fallback** — nilai fiktif TIDAK PERNAH ditulis ke kolom hasil. Horizons
   kadang membalas `HTTP 503` (server sibuk) — kadang cuma sesekali, kadang beruntun/"badai"
   selama beberapa menit saat server sedang overload (di luar kendali notebook ini). Klien di
   Bagian 4 menangani ini dengan **3 lapis pertahanan**:
   1. **Retry otomatis** sampai 6x per query dengan jeda membesar (1s, 2s, 4s, 8s, 16s, 30s).
   2. **Toleran-gagal**: kalau satu batch/bracket tetap gagal setelah 6x retry, batch/bracket
      itu **dilewati** (bukan menjatuhkan seluruh sel) — dicatat sebagai `[PERINGATAN]`, dan
      cakupannya bisa dilengkapi dengan **menjalankan ulang sel yang sama**.
   3. **Cache ke disk** (Bagian 4) — request yang SUDAH berhasil disimpan ke folder
      `horizons_cache/` di Colab, jadi kalau kamu menjalankan ulang sel setelah gagal sebagian,
      bagian yang sudah berhasil TIDAK di-query ulang — hanya bagian yang tadi gagal yang
      dicoba lagi. Ini membuat proses **bisa dilanjutkan bertahap** kalau Horizons sedang tidak
      stabil, alih-alih harus mengulang dari nol tiap kali.
   Diperlukan koneksi internet aktif di runtime Colab.
2. **Kalau `[PERINGATAN]` HTTP 503 muncul sangat banyak/beruntun** (bukan sesekali), itu
   biasanya tandanya Horizons sedang membatasi/menolak trafik dari IP Colab secara umum (IP
   Colab dipakai bergantian oleh banyak pengguna Google sekaligus) — bukan sesuatu yang bisa
   diperbaiki dari sisi kode. Solusinya: **jalankan ulang sel yang sama beberapa kali** (cache
   di poin 1.3 membuat tiap percobaan makin cepat karena yang sudah berhasil tidak diulang),
   atau tunggu beberapa menit lalu coba lagi.
3. **Estimasi waktu jalan.** Fase 1 memindai dengan langkah 6 jam sepanjang seluruh rentang
   tahun (identik dengan `SCAN_STEP_MS` produksi) — untuk 2017–2026 ini berarti ±14.600 titik
   waktu awal + refinement Newton-Raphson per bulan sinodis. **Disarankan uji coba dulu dengan
   1 tahun** (mis. `FROM_YEAR = TO_YEAR = 2024`) sebelum menjalankan rentang penuh 10 tahun.
   Fase 2 (per tahun: 1 pemindaian grid saksi KHGT + pipeline lokal Wujudul Hilal) juga
   memakan beberapa menit untuk 10 tahun.
4. **Substitusi library non-inti** (tidak memengaruhi hasil astronomis):
   - `tz-lookup` (Node) → `timezonefinder` (Python), untuk menentukan zona waktu tiap titik
     grid saksi KHGT dari koordinat lat/lon.
   - `luxon` (Node) → `datetime` + `pytz` bawaan Python.
   - Concurrency: produksi memakai semaphore 4 request paralel (`MAX_CONCURRENCY=4`) di
     Node.js; notebook ini memakai `ThreadPoolExecutor(max_workers=2)` — sengaja **lebih
     rendah** dari produksi supaya tidak membebani Horizons dari IP Colab yang sudah dipakai
     banyak pengguna sekaligus (mengurangi risiko `HTTP 503`). Hasil astronomis identik, hanya
     kecepatan yang berbeda; bisa dinaikkan lagi di Bagian 4 kalau koneksi sedang stabil.


## Peta Rujukan Kode Sumber (untuk Lampiran Skripsi)

Setiap sel kode di bawah mencantumkan file asal (docstring) di aplikasi produksi
(`International Astronomical Studies/src/...`) yang menjadi rujukan porting-nya:

| Bagian Notebook | File Sumber TypeScript |
|---|---|
| Utilitas sudut | `src/lib/mathAngle.ts` |
| Julian Date | `src/lib/horizonsClient.ts` (`dateToJD`, `jdToDate`) |
| Klien API Horizons | `src/lib/horizonsClient.ts` (`queryHorizons`, `parseSOE`) |
| Query builder Horizons | `src/lib/horizonsQueries.ts` |
| Waktu Matahari terbenam | `src/lib/sunset.ts` (porting `suncalc` npm) |
| Perhitungan geosentris | `src/lib/geoCalc.ts` |
| Newton-Raphson konjungsi | `src/lib/newMoonNR.ts` |
| Rule A/B (Wujudul Hilal) & KHGT | `src/lib/wujudulHilalRule.ts`, `src/lib/khgtRule.ts` |
| Estimator tanggal konjungsi | `src/lib/khgtPipeline.ts` (`estimateRamadanConjDate`), `src/lib/ramadanFromSyaban.ts` (`estimateRamadan1`) |
| **Fase 1** — Evaluasi Konjungsi | `src/app/api/konjungsi-periode/route.ts` |
| Pipeline KHGT global (grid saksi) | `src/lib/khgtPipeline.ts` |
| Pipeline lokal Wujudul Hilal | `src/lib/ramadanFromSyaban.ts` |
| Data historis resmi Indonesia | `src/lib/officialHistory/seedIndonesia.ts`, `src/lib/officialHistory/resolve.ts` |
| **Fase 2** — Evaluasi History Global dan Local | `src/app/api/evaluate/route.ts` |


## 0. Instalasi & Import

In [ ]:
!pip -q install requests pytz timezonefinder python-dateutil pandas openpyxl

In [ ]:
"""Import & tipe data bersama untuk seluruh notebook."""
import os
import time
import math
import random
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import cmp_to_key
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Optional, List, Dict, Tuple, Any, Callable

import requests
import pytz
import pandas as pd
from dateutil import parser as dtparser

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


## 0.1 Utilitas Log Progres

Setiap tahap proses di notebook ini mencetak baris status **sebelum** mulai (`>>`) dan
**sesudah** selesai (`[OK]` / `[GAGAL]` / `[PERINGATAN]`), supaya sel yang berjalan lama
(pemindaian Newton-Raphson, pemindaian grid saksi KHGT, dll.) tidak terasa "diam" tanpa
kejelasan apakah masih berjalan, berhasil, atau macet.

In [ ]:
def log_step(msg: str) -> None:
    """Tandai sebuah tahap MULAI dikerjakan."""
    print(f">> {msg}", flush=True)


def log_ok(msg: str) -> None:
    """Tandai sebuah tahap BERHASIL diselesaikan."""
    print(f"   [OK] {msg}", flush=True)


def log_warn(msg: str) -> None:
    """Tandai peringatan non-fatal (mis. sedang mencoba ulang)."""
    print(f"   [PERINGATAN] {msg}", flush=True)


def log_fail(msg: str) -> None:
    """Tandai sebuah tahap GAGAL (tetap lanjut, bukan menghentikan seluruh proses)."""
    print(f"   [GAGAL] {msg}", flush=True)


## 1. Konfigurasi Lokasi & Rentang Tahun

Lokasi default adalah **Kota Bekasi, Jawa Barat** — lokasi pengamatan utama skripsi
(lat/lon identik dengan default `/api/konjungsi-periode` pada aplikasi produksi).
Ubah `FROM_YEAR` / `TO_YEAR` / `LAT` / `LON` / `TZ` di sel ini untuk menjalankan skenario lain;
seluruh sel eksekusi Fase 1 dan Fase 2 di bagian bawah notebook memakai variabel ini.

In [ ]:
log_step("Mengatur periode tahun & lokasi pengamatan")

# Rentang tahun evaluasi (skripsi Bab IV: 2017-2026, 10 tahun penuh)
FROM_YEAR = 2017
TO_YEAR = 2026

# Lokasi pengamatan utama: Kota Bekasi, Jawa Barat
LAT = -6.2349
LON = 107.0000
TZ = "Asia/Jakarta"

log_ok(f"Periode tahun berhasil diatur: {FROM_YEAR}-{TO_YEAR}")
log_ok(f"Lokasi pengamatan berhasil diatur: Kota Bekasi (lat={LAT}, lon={LON}, tz={TZ})")


## 2. Utilitas Sudut

Porting langsung dari `src/lib/mathAngle.ts`. `wrap_to_180` dan `wrap_to_360` sengaja
mempertahankan koreksi dua-langkah (`if r >= ...: r -= 360`) walau operator modulo Python dan
JavaScript berbeda tanda untuk bilangan negatif — dengan koreksi ini kedua bahasa selalu
konvergen ke representasi sudut kanonis yang sama.

In [ ]:
# ======================================================================
# PERSAMAAN (3.2) - NORMALISASI SUDUT | wrapTo180
# LANGKAH 5 dari 14 -> f(t) = wrapTo180( d-lambda(t) )
# Hasil dipaksa ke rentang -180 s.d. 180 derajat agar sudut tidak
# 'meloncat' saat melewati batas 0/360 derajat.
# ======================================================================
def wrap_to_180(deg: float) -> float:
    """Wrap angle to [-180, 180) degrees. Port of mathAngle.ts:wrapTo180."""
    r = deg % 360
    if r >= 180:
        r -= 360
    if r < -180:
        r += 360
    return r


def wrap_to_360(deg: float) -> float:
    """Wrap angle to [0, 360) degrees. Port of mathAngle.ts:wrapTo360."""
    r = deg % 360
    if r < 0:
        r += 360
    return r


def deg_to_rad(deg: float) -> float:
    return deg * math.pi / 180


def rad_to_deg(rad: float) -> float:
    return rad * 180 / math.pi


## 3. Julian Date

Porting dari `src/lib/horizonsClient.ts` (`dateToJD`, `jdToDate`). Formula standar Meeus
untuk konversi kalender Gregorian <-> Julian Date, dipertahankan identik termasuk pembagi
milidetik pada `date_to_jd` (kontribusinya terhadap JD dapat diabaikan, ~1e-9 hari, dan
dipertahankan agar hasil numerik 100% sama dengan produksi).

In [ ]:
def date_to_jd(dt: datetime) -> float:
    """UTC datetime -> Julian Date. Port of horizonsClient.ts:dateToJD."""
    y = dt.year
    m = dt.month
    d = (dt.day + dt.hour / 24 + dt.minute / 1440 + dt.second / 86400
         + dt.microsecond / 86400000000)  # matches TS divisor exactly (ms handled as 1/1000 sec of a day-scaled term)

    yr, mo = y, m
    if mo <= 2:
        yr -= 1
        mo += 12

    a = math.floor(yr / 100)
    b = 2 - a + math.floor(a / 4)
    return math.floor(365.25 * (yr + 4716)) + math.floor(30.6001 * (mo + 1)) + d + b - 1524.5


def jd_to_date(jd: float) -> datetime:
    """Julian Date -> UTC datetime. Port of horizonsClient.ts:jdToDate."""
    J = jd + 0.5
    Z = math.floor(J)
    F = J - Z
    if Z < 2299161:
        A = Z
    else:
        alpha = math.floor((Z - 1867216.25) / 36524.25)
        A = Z + 1 + alpha - math.floor(alpha / 4)
    B = A + 1524
    C = math.floor((B - 122.1) / 365.25)
    D = math.floor(365.25 * C)
    E = math.floor((B - D) / 30.6001)

    day_frac = B - D - math.floor(30.6001 * E) + F
    day = math.floor(day_frac)
    month = E - 1 if E < 14 else E - 13
    year = C - 4716 if month > 2 else C - 4715

    hrs = (day_frac - day) * 24
    h = math.floor(hrs)
    minute = math.floor((hrs - h) * 60)
    sec = round(((hrs - h) * 3600 - minute * 60) * 1000) / 1000
    micro = round((sec % 1) * 1_000_000)
    # Built via timedelta (not the datetime(...) constructor) because rounding
    # can push sec to exactly 60.0 at the boundary — JS's Date.UTC() silently
    # rolls that over into the next minute, but Python's datetime() constructor
    # rejects second=60 outright. timedelta addition rolls over the same way
    # Date.UTC does, keeping the two languages' outputs identical.
    base = datetime(int(year), int(month), int(day), tzinfo=timezone.utc)
    return base + timedelta(hours=int(h), minutes=int(minute),
                             seconds=int(math.floor(sec)), microseconds=int(micro))


## 4. Klien API NASA/JPL Horizons

Porting dari `src/lib/horizonsClient.ts` (`queryHorizons`, `parseSOE`). Perbedaan yang
disengaja dari produksi (semuanya menambah ketangguhan, bukan mengubah logika astronomis):

- **Tidak ada cabang mock/fallback** — kebijakan data akademik sistem ini menyatakan data mock
  TIDAK VALID untuk Bab IV, jadi nilai fiktif TIDAK PERNAH ditulis ke kolom hasil.
- **Retry berlapis**: sampai `MAX_RETRIES` kali per query dengan jeda eksponensial + *jitter*
  sebelum satu query dianggap gagal.
- **Cache ke DISK** (folder `horizons_cache/`), bukan cuma di memori — bertahan selintas sesi
  Colab yang sama walau selnya dijalankan ulang berkali-kali. Query yang paramnya identik
  dengan yang sudah pernah sukses TIDAK dikirim ulang ke NASA sama sekali.
- **Eksekusi konkuren yang toleran-gagal** (`_run_concurrent_tolerant`, dipakai Bagian 6 & 12):
  kalau sebagian batch/bracket gagal walau sudah di-retry maksimal, sisanya tetap diproses dan
  hasil yang gagal cukup ditandai `[PERINGATAN]` — bukan menjatuhkan seluruh scan yang sudah
  berjalan bermenit-menit. Jalankan ulang sel yang sama untuk melengkapi cakupan (lihat cache
  di atas — bagian yang sudah berhasil tidak diulang).

**Opsional: cache lintas-sesi via Google Drive.** Folder `horizons_cache/` di atas hilang kalau
runtime Colab benar-benar disconnect (bukan cuma re-run sel). Untuk cache yang bertahan lintas
sesi, mount Google Drive lalu ubah `CACHE_DIR` di sel ini menjadi path di Drive, misalnya:
```python
from google.colab import drive
drive.mount('/content/drive')
CACHE_DIR = '/content/drive/MyDrive/horizons_cache'
```

In [ ]:
HORIZONS_URL = "https://ssd.jpl.nasa.gov/api/horizons.api"
USER_AGENT = "InternationalAstronomicalStudies-Colab/1.0 (academic research reproduction)"

SUN_CMD = "'10'"
MOON_CMD = "'301'"

MAX_RETRIES = 6
RETRY_BASE_DELAY = 1.0   # seconds, doubles each attempt (capped at RETRY_MAX_DELAY)
RETRY_MAX_DELAY = 30.0

# Disk cache directory — see Section 4 markdown for mounting Google Drive here
# for cross-session persistence instead of the default (per-VM, ephemeral) path.
CACHE_DIR = "horizons_cache"

_mem_cache: Dict[str, str] = {}
_scan_stats = {"liveCount": 0, "cacheCount": 0, "diskCacheCount": 0, "failedCount": 0, "retryCount": 0}

# Shared bounded executor. Production uses MAX_CONCURRENCY=4; this notebook
# deliberately uses fewer workers (see "Catatan Penting" point 4) to reduce
# load on Horizons from Colab's shared IP range. Raise back to 4 if your
# connection is stable and you want more speed.
_executor = ThreadPoolExecutor(max_workers=2)


def _run_concurrent(fns: List[Callable[[], Any]], progress_label: Optional[str] = None) -> List[Any]:
    """Run `fns` on the shared executor, preserving input order. Raises on the
    first task that raises (after its own internal retries are exhausted) —
    use _run_concurrent_tolerant below wherever a handful of failed
    sub-queries should not abort an entire multi-minute scan."""
    total = len(fns)
    futures = {_executor.submit(fn): i for i, fn in enumerate(fns)}
    results: List[Any] = [None] * total
    done = 0
    step = max(1, total // 10)
    for fut in as_completed(futures):
        idx = futures[fut]
        results[idx] = fut.result()
        done += 1
        if progress_label and (done % step == 0 or done == total):
            log_step(f"{progress_label}: {done}/{total} selesai")
    return results


def _run_concurrent_tolerant(fns: List[Callable[[], Any]],
                              progress_label: Optional[str] = None) -> Tuple[List[Any], List[Tuple[int, Exception]]]:
    """Like _run_concurrent, but a failing task (Horizons still down after
    MAX_RETRIES) does NOT abort the rest — its slot in the returned results
    list is None, and (index, exception) is appended to the returned failures
    list instead. Lets scan_for_brackets / find_conjunctions_in_range / etc.
    finish the batches that DID succeed and report exactly which ones didn't,
    rather than losing everything to one persistently-failing request."""
    total = len(fns)
    futures = {_executor.submit(fn): i for i, fn in enumerate(fns)}
    results: List[Any] = [None] * total
    failures: List[Tuple[int, Exception]] = []
    done = 0
    step = max(1, total // 10)
    for fut in as_completed(futures):
        idx = futures[fut]
        try:
            results[idx] = fut.result()
        except Exception as e:
            failures.append((idx, e))
        done += 1
        if progress_label and (done % step == 0 or done == total):
            fail_note = f" ({len(failures)} gagal sejauh ini)" if failures else ""
            log_step(f"{progress_label}: {done}/{total} selesai{fail_note}")
    return results, failures


def _cache_key(params: Dict[str, str]) -> str:
    s = "&".join(f"{k}={v}" for k, v in sorted(params.items()))
    return hashlib.sha1(s.encode()).hexdigest()


def _disk_cache_path(key: str) -> str:
    return os.path.join(CACHE_DIR, f"{key}.txt")


def _load_disk_cache(key: str) -> Optional[str]:
    path = _disk_cache_path(key)
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception:
        return None  # corrupt/unreadable cache file — treat as a cache miss, re-fetch


def _save_disk_cache(key: str, result: str) -> None:
    try:
        os.makedirs(CACHE_DIR, exist_ok=True)
        with open(_disk_cache_path(key), "w", encoding="utf-8") as f:
            f.write(result)
    except Exception:
        pass  # best-effort only — a cache write failure must never break the query itself


def _fetch_horizons_once(params: Dict[str, str], timeout_s: float) -> str:
    q = dict(params)
    q["format"] = "json"
    resp = requests.get(HORIZONS_URL, params=q, headers={"User-Agent": USER_AGENT}, timeout=timeout_s)
    if resp.status_code != 200:
        raise RuntimeError(f"HORIZONS HTTP {resp.status_code}")
    data = resp.json()
    if data.get("error"):
        raise RuntimeError(f"HORIZONS API error: {data['error']}")
    return data["result"]


def query_horizons(params: Dict[str, str]) -> str:
    """Live-only Horizons query with memory+disk cache and up to MAX_RETRIES
    retries (exponential backoff + jitter) on transient errors. Port of
    horizonsClient.ts:queryHorizons WITHOUT the mock/fallback branch — if every
    retry is exhausted, this raises (no fabricated data); the disk cache means
    a subsequent re-run of the same cell never re-fetches what already
    succeeded (see Section 4 markdown)."""
    key = _cache_key(params)
    if key in _mem_cache:
        _scan_stats["cacheCount"] += 1
        return _mem_cache[key]

    disk_hit = _load_disk_cache(key)
    if disk_hit is not None:
        _mem_cache[key] = disk_hit
        _scan_stats["diskCacheCount"] += 1
        return disk_hit

    last_err: Optional[Exception] = None
    delay = RETRY_BASE_DELAY
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = _fetch_horizons_once(params, 30 if attempt == 1 else 15)
            _mem_cache[key] = result
            _save_disk_cache(key, result)
            _scan_stats["liveCount"] += 1
            return result
        except Exception as e:
            last_err = e
            msg = str(e)
            transient = (any(code in msg for code in ("502", "503", "504"))
                         or "Timeout" in msg or "timed out" in msg or "Connection" in msg)
            if not transient or attempt == MAX_RETRIES:
                _scan_stats["failedCount"] += 1
                break
            _scan_stats["retryCount"] += 1
            wait = min(delay, RETRY_MAX_DELAY) + random.uniform(0, 1)
            log_warn(f"HORIZONS {msg} — mencoba ulang ({attempt}/{MAX_RETRIES}) dalam {wait:.1f}s...")
            time.sleep(wait)
            delay *= 2

    raise RuntimeError(f"HORIZONS query gagal setelah {MAX_RETRIES} percobaan: {last_err}")


def parse_soe(result: str) -> List[Tuple[str, List[float]]]:
    """Parse $$SOE..$$EOE block from a HORIZONS result string.
    Port of horizonsClient.ts:parseSOE."""
    soe = result.find("$$SOE")
    eoe = result.find("$$EOE")
    if soe == -1 or eoe == -1:
        raise RuntimeError("Could not find $$SOE/$$EOE markers in HORIZONS response")
    block = result[soe + 5:eoe].strip()
    parsed = []
    for line in block.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 2:
            continue
        date_str = parts[0]
        values = []
        for p in parts[1:]:
            try:
                values.append(float(p))
            except ValueError:
                pass
        parsed.append((date_str, values))
    return parsed


## 5. Query Builder Horizons

Porting dari `src/lib/horizonsQueries.ts`. Tiga jenis query dipakai di seluruh pipeline:

- `get_ecliptic_lon` — bujur ekliptika geosentris (`QUANTITIES='31'`), dipakai Newton-Raphson.
- `get_geocentric_apparent_radec` — RA/Dec geosentris apparent (`QUANTITIES='2'`), dipakai
  perhitungan elongasi & altitude geosentris (KHGT).
- `get_topo_azel` — azimuth/altitude topocentris dari titik pengamatan tertentu
  (`QUANTITIES='4'`), dipakai Rule B (Wujudul Hilal) dan pengamatan saksi KHGT.

Ketiganya mengirim seluruh epoch dalam **satu** request Horizons (`TLIST`) dan mengasumsikan
baris hasil `$$SOE` kembali dalam urutan epoch yang sama seperti yang dikirim (Horizons
mengembalikan baris terurut kronologis) — sama seperti asumsi kode produksi; oleh karena itu
semua epoch yang dikirim ke fungsi ini harus sudah terurut menaik, seperti pada kode aslinya.

In [ ]:
# ======================================================================
# LANGKAH 3 dari 14 - AMBIL BUJUR EKLIPTIKA DARI NASA JPL HORIZONS
# DATA GEOSENTRIS (CENTER 500@399) -> bahan Persamaan (3.1)
# QUANTITIES '31' = ObsEcLon. Epoch dikirim sebagai TLIST (daftar
# waktu diskrit), BUKAN deret waktu tetap.
# ======================================================================
def get_ecliptic_lon(command: str, epochs_utc: List[datetime]) -> List[float]:
    """COMMAND '10' (Sun) or '301' (Moon). Returns ObsEcLon (deg) per epoch, same order.
    Port of horizonsQueries.ts:getEclipticLon."""
    tlist = ", ".join(f"{date_to_jd(d):.8f}" for d in epochs_utc)
    params = {
        "COMMAND": command, "EPHEM_TYPE": "'OBSERVER'", "CENTER": "'500@399'",
        "QUANTITIES": "'31'", "TIME_TYPE": "'UT'", "TIME_DIGITS": "'SECONDS'",
        "ANG_FORMAT": "'DEG'", "EXTRA_PREC": "'YES'", "APPARENT": "'AIRLESS'",
        "CSV_FORMAT": "'YES'", "CAL_TYPE": "'GREGORIAN'", "TLIST": f"'{tlist}'",
    }
    parsed = parse_soe(query_horizons(params))
    return [row[1][0] for row in parsed]


# ======================================================================
# DATA TOPOSENTRIS - ALTITUDE & AZIMUT DI LOKASI PENGAMAT
# Dipakai RULE B (Pers. 3.10): altitude Bulan saat sunset > 0 derajat
# CENTER 'coord@399' + SITE_COORD = koordinat Kota Bekasi
# ======================================================================
def get_topo_azel(command: str, epochs_utc: List[datetime], lat: float, lon: float,
                   alt_km: float = 0.0) -> List[Tuple[float, float]]:
    """Returns list of (az_deg, el_deg) per epoch, same order.
    Port of horizonsQueries.ts:getTopoAzEl."""
    tlist = ", ".join(f"{date_to_jd(d):.8f}" for d in epochs_utc)
    params = {
        "COMMAND": command, "EPHEM_TYPE": "'OBSERVER'", "CENTER": "'coord@399'",
        "COORD_TYPE": "'GEODETIC'", "SITE_COORD": f"'{lon},{lat},{alt_km}'",
        "QUANTITIES": "'4'", "TIME_TYPE": "'UT'", "TIME_DIGITS": "'SECONDS'",
        "ANG_FORMAT": "'DEG'", "EXTRA_PREC": "'YES'", "APPARENT": "'AIRLESS'",
        "CSV_FORMAT": "'YES'", "CAL_TYPE": "'GREGORIAN'", "TLIST": f"'{tlist}'",
    }
    parsed = parse_soe(query_horizons(params))
    return [(row[1][0], row[1][1]) for row in parsed]


def get_geocentric_apparent_radec(command: str, epochs_utc: List[datetime]) -> List[Tuple[float, float]]:
    """Returns list of (ra_deg, dec_deg) per epoch, same order.
    Port of horizonsQueries.ts:getGeocentricApparentRADec."""
    tlist = ", ".join(f"{date_to_jd(d):.8f}" for d in epochs_utc)
    params = {
        "COMMAND": command, "EPHEM_TYPE": "'OBSERVER'", "CENTER": "'500@399'",
        "QUANTITIES": "'2'", "TIME_TYPE": "'UT'", "TIME_DIGITS": "'SECONDS'",
        "ANG_FORMAT": "'DEG'", "EXTRA_PREC": "'YES'", "APPARENT": "'AIRLESS'",
        "CSV_FORMAT": "'YES'", "CAL_TYPE": "'GREGORIAN'", "TLIST": f"'{tlist}'",
    }
    parsed = parse_soe(query_horizons(params))
    return [(row[1][0], row[1][1]) for row in parsed]


## 6. Waktu Matahari Terbenam (porting `suncalc`)

Aplikasi produksi memakai library npm `suncalc` untuk menghitung waktu Matahari terbenam.
Sel ini adalah **porting baris-per-baris** algoritma `suncalc.js` (formula astronomi dari
aa.quae.nl, digunakan luas dan gratis/open-source) ke Python, agar hasil numerik identik
sampai presisi floating-point — bukan library pengganti dengan formula berbeda.

`get_sunset` mem-parsing tanggal pada **tengah hari (jam 12:00) waktu lokal**, bukan tengah
malam, persis seperti `sunset.ts:getSunset` — supaya SunCalc tidak salah mengambil sunset hari
sebelumnya untuk lokasi di belahan bumi barat.

In [ ]:
# ======================================================================
# WAKTU MATAHARI TERBENAM (SUNSET) - porting library suncalc
# Menghasilkan t_sunset yang dipakai:
#   RULE A (Pers. 3.9) : t_konjungsi < t_sunset
#   RULE B (Pers. 3.10): altitude Bulan DIUKUR PADA t_sunset
# Sunset dihitung dari tanggal + koordinat lokasi (bukan query Horizons).
# ======================================================================
_RAD = math.pi / 180
_OBLIQUITY = _RAD * 23.4397  # obliquity of the Earth
_DAY_MS = 86400000
_J1970 = 2440588
_J2000 = 2451545
_J0 = 0.0009

_SUNCALC_TIMES = [
    (-0.833, "sunrise", "sunset"),
    (-0.3, "sunriseEnd", "sunsetStart"),
    (-6, "dawn", "dusk"),
    (-12, "nauticalDawn", "nauticalDusk"),
    (-18, "nightEnd", "night"),
    (6, "goldenHourEnd", "goldenHour"),
]


def _to_julian(dt: datetime) -> float:
    return dt.timestamp() * 1000 / _DAY_MS - 0.5 + _J1970


def _from_julian(j: float) -> datetime:
    ms = (j + 0.5 - _J1970) * _DAY_MS
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc)


def _to_days(dt: datetime) -> float:
    return _to_julian(dt) - _J2000


def _solar_mean_anomaly(d: float) -> float:
    return _RAD * (357.5291 + 0.98560028 * d)


def _ecliptic_longitude(M: float) -> float:
    C = _RAD * (1.9148 * math.sin(M) + 0.0200 * math.sin(2 * M) + 0.0003 * math.sin(3 * M))
    P = _RAD * 102.9372
    return M + C + P + math.pi


def _declination(l: float, b: float) -> float:
    return math.asin(math.sin(b) * math.cos(_OBLIQUITY) + math.cos(b) * math.sin(_OBLIQUITY) * math.sin(l))


def _julian_cycle(d: float, lw: float) -> float:
    return round(d - _J0 - lw / (2 * math.pi))


def _approx_transit(Ht: float, lw: float, n: float) -> float:
    return _J0 + (Ht + lw) / (2 * math.pi) + n


def _solar_transit_j(ds: float, M: float, L: float) -> float:
    return _J2000 + ds + 0.0053 * math.sin(M) - 0.0069 * math.sin(2 * L)


def _hour_angle(h: float, phi: float, d: float) -> float:
    cos_h = (math.sin(h) - math.sin(phi) * math.sin(d)) / (math.cos(phi) * math.cos(d))
    cos_h = max(-1.0, min(1.0, cos_h))  # guard float noise only (JS acos(>1)=NaN)
    return math.acos(cos_h)


def _observer_angle(height: float) -> float:
    return -2.076 * math.sqrt(height) / 60


def _get_set_j(h, lw, phi, dec, n, M, L):
    w = _hour_angle(h, phi, dec)
    a = _approx_transit(w, lw, n)
    return _solar_transit_j(a, M, L)


def suncalc_get_times(date_utc: datetime, lat: float, lon: float, height: float = 0.0) -> Dict[str, Optional[datetime]]:
    """Port of suncalc.js getTimes(). `date_utc` should be a tz-aware UTC instant
    representing the reference time-of-day for the target civil day (noon for
    get_sunset below; see get_nz_fajr_nightend_utc for the one deliberate
    exception, matching production)."""
    lw = _RAD * -lon
    phi = _RAD * lat
    dh = _observer_angle(height)
    d = _to_days(date_utc)
    n = _julian_cycle(d, lw)
    ds = _approx_transit(0, lw, n)
    M = _solar_mean_anomaly(ds)
    L = _ecliptic_longitude(M)
    dec = _declination(L, 0)
    j_noon = _solar_transit_j(ds, M, L)

    result: Dict[str, Optional[datetime]] = {
        "solarNoon": _from_julian(j_noon),
        "nadir": _from_julian(j_noon - 0.5),
    }
    for angle, rise_name, set_name in _SUNCALC_TIMES:
        h0 = (angle + dh) * _RAD
        j_set = _get_set_j(h0, lw, phi, dec, n, M, L)
        j_rise = j_noon - (j_set - j_noon)
        result[rise_name] = _from_julian(j_rise)
        result[set_name] = _from_julian(j_set)
    return result


def get_sunset(date_str: str, lat: float, lon: float, tz: str) -> Dict[str, Any]:
    """Port of sunset.ts:getSunset. `date_str` = 'YYYY-MM-DD' civil date, `tz` = IANA zone."""
    zone = pytz.timezone(tz)
    naive_noon = datetime.strptime(date_str, "%Y-%m-%d").replace(hour=12, minute=0, second=0, microsecond=0)
    local_noon = zone.localize(naive_noon)
    utc_noon = local_noon.astimezone(timezone.utc)

    times = suncalc_get_times(utc_noon, lat, lon)
    sunset_utc = times.get("sunset")
    if sunset_utc is None:
        raise RuntimeError(f"Could not compute sunset for {date_str} at lat={lat} lon={lon}")

    sunset_local = sunset_utc.astimezone(zone)
    return {"sunsetUTC": sunset_utc, "sunsetLocal": sunset_local.isoformat(), "timezone": tz}


NZ_LAT, NZ_LON, NZ_TZ = -41.2866, 174.7756, "Pacific/Auckland"


def get_nz_fajr_nightend_utc(date_iso: str) -> datetime:
    """Astronomical dawn (nightEnd, sun alt ~ -18 deg) at Wellington NZ, for the NZ
    local civil date AFTER the conjunction instant `date_iso`. Used by KHGT PKG2(a).
    Port of sunset.ts:getNzFajrNightEndUTC — deliberately uses NZ local MIDNIGHT
    (not noon) as the SunCalc reference instant, exactly like production: safe
    here because Wellington's positive longitude means local-midnight->UTC never
    crosses into the wrong UT day (the noon-fix in get_sunset() above exists only
    for western/negative-longitude cases)."""
    dt_utc = dtparser.isoparse(date_iso)
    if dt_utc.tzinfo is None:
        dt_utc = dt_utc.replace(tzinfo=timezone.utc)
    nz_zone = pytz.timezone(NZ_TZ)
    nz_dt = dt_utc.astimezone(nz_zone)
    nz_date_str = nz_dt.strftime("%Y-%m-%d")
    next_nz_midnight_naive = datetime.strptime(nz_date_str, "%Y-%m-%d") + timedelta(days=1)
    next_nz_midnight = nz_zone.localize(next_nz_midnight_naive)
    utc_instant = next_nz_midnight.astimezone(timezone.utc)

    times = suncalc_get_times(utc_instant, NZ_LAT, NZ_LON)
    night_end = times.get("nightEnd")
    if night_end is None:
        raise RuntimeError(f"Could not compute nightEnd for NZ on {nz_date_str}")
    return night_end


## 7. Perhitungan Geosentris

Porting dari `src/lib/geoCalc.ts`: GMST, Local Sidereal Time, altitude geosentris, dan
elongasi geosentris Bulan-Matahari — parameter inti KHGT.

In [ ]:
def gmst_deg(date_utc: datetime) -> float:
    """Greenwich Mean Sidereal Time (deg). Port of geoCalc.ts:gmstDeg."""
    jd = date_utc.timestamp() / 86400 + 2440587.5
    T = (jd - 2451545.0) / 36525.0
    gmst = (280.46061837 + 360.98564736629 * (jd - 2451545.0)
            + 0.000387933 * T * T - (T ** 3) / 38710000.0)
    return wrap_to_360(gmst)


def local_sidereal_deg(gmst: float, lon_deg: float) -> float:
    return wrap_to_360(gmst + lon_deg)


def geocentric_alt_deg(ra_deg: float, dec_deg: float, lat_deg: float, lon_deg: float,
                        date_utc: datetime) -> float:
    """Geocentric altitude of a body (KHGT metric). Port of geoCalc.ts:geocentricAltDeg."""
    lst = local_sidereal_deg(gmst_deg(date_utc), lon_deg)
    ha = deg_to_rad(wrap_to_360(lst - ra_deg))
    lat = deg_to_rad(lat_deg)
    dec = deg_to_rad(dec_deg)
    sin_alt = math.sin(dec) * math.sin(lat) + math.cos(dec) * math.cos(lat) * math.cos(ha)
    sin_alt = max(-1.0, min(1.0, sin_alt))  # guard float noise only; TS leaves unclamped
    return rad_to_deg(math.asin(sin_alt))


def geocentric_elong_deg(ra_moon_deg: float, dec_moon_deg: float,
                          ra_sun_deg: float, dec_sun_deg: float) -> float:
    """Geocentric Moon-Sun elongation. Port of geoCalc.ts:geocentricElongDeg."""
    ram, decm = deg_to_rad(ra_moon_deg), deg_to_rad(dec_moon_deg)
    ras, decs = deg_to_rad(ra_sun_deg), deg_to_rad(dec_sun_deg)
    cos_e = math.sin(decm) * math.sin(decs) + math.cos(decm) * math.cos(decs) * math.cos(ram - ras)
    cos_e = max(-1.0, min(1.0, cos_e))
    return rad_to_deg(math.acos(cos_e))


## 8. Newton-Raphson Conjunction Finder

Porting dari `src/lib/newMoonNR.ts` — inti algoritma skripsi. Fungsi target:

$$f(t) = \mathrm{wrapTo180}\big(\lambda_{\text{Bulan}}(t) - \lambda_{\text{Matahari}}(t)\big), \quad f(t) = 0$$

Turunan didekati beda-tengah (*central difference*): $f'(t) \approx \dfrac{f(t+\delta)-f(t-\delta)}{2\delta}$,
$\delta = 60$ detik. Kriteria berhenti iterasi (sesuai Bab III skripsi): `|f(tn)| <= 1e-6°`
ATAU (`|stepSec| <= 0.2 detik` DAN `|f(tn)| < 0.01°`), maksimum 30 iterasi. Pemindaian awal
(*bracketing*) memakai langkah 6 jam sepanjang jendela waktu; pasangan titik dengan pergantian
tanda `f(t)` yang salah satu titiknya berada di dekat oposisi (`|f|>90°`, bulan purnama, BUKAN
konjungsi) disaring keluar sebelum masuk ke Newton-Raphson.

In [ ]:
# ======================================================================
# INTI ALGORITMA - PARAMETER & AMBANG NEWTON-RAPHSON (Bab III.3)
# ======================================================================
DELTA_S = 60          # delta = 60 detik -> PERSAMAAN (3.7) central difference
EPS_ANGLE = 1e-6      # epsilon sudut 1e-6 derajat -> AMBANG RESIDUAL ABSOLUT |f(tn)|
EPS_TIME = 0.2        # epsilon waktu 0,2 detik -> AMBANG ABSOLUTE APPROXIMATE ERROR |stepSec|
MAX_ITER = 30         # batas maksimum iterasi (pencegah proses tanpa batas)
SCAN_STEP = timedelta(hours=6)         # LANGKAH 2 - interval pemindaian awal 6 jam
BATCH_SIZE = 40                        # maks 40 epoch per permintaan Horizons
DEDUP_THRESHOLD = timedelta(hours=12)  # AMBANG DEDUPLIKASI hasil konjungsi (12 jam)

# ======================================================================
# STATISTIK TAHAP DATA PREPARATION & MODELING (Tabel 4.4 & 4.6 Bab IV)
# Dicatat sebagai efek-samping oleh scan_for_brackets/try_nr_on_bracket di
# bawah -- TIDAK mengubah nilai balik atau perilaku fungsi manapun, jadi
# Fase 1/Fase 2 di atas tetap berjalan identik seperti sebelumnya.
# ======================================================================
_conjunction_scan_stats = {
    "totalEpochs": 0, "totalRawValues": 0, "totalSignChanges": 0,
    "filteredAsOpposition": 0, "bracketsFormed": 0,
    "totalNRIterations": 0, "bracketsConverged": 0, "bracketsFallback": 0,
}


def reset_conjunction_scan_stats() -> None:
    """Panggil sebelum scan baru (mis. sebelum run_phase1) agar statistik
    Tabel 4.4/4.6 hanya menghitung scan yang sedang berjalan, bukan akumulasi
    lintas-run."""
    for k in _conjunction_scan_stats:
        _conjunction_scan_stats[k] = 0


@dataclass
class ConjSimple:
    t: datetime
    iso: str
    converged: bool


def _iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


# ======================================================================
# FUNGSI TARGET f(t) - PERSAMAAN (3.1) + (3.2)
# LANGKAH 4 & 5 dari 14
# ======================================================================
def _eval_f_batch_detailed(epochs: List[datetime]) -> List[Dict[str, float]]:
    """f(t) = wrapTo180(moonEcLon - sunEcLon) for a batch of epochs, plus raw
    components for audit. Port of newMoonNR.ts:evalFBatchDetailed."""
    moon_ecl = get_ecliptic_lon(MOON_CMD, epochs)
    sun_ecl = get_ecliptic_lon(SUN_CMD, epochs)
    out = []
    for i, ep in enumerate(epochs):
        # --- PERSAMAAN (3.1) - SELISIH BUJUR EKLIPTIKA -------------------
        #     d-lambda(t) = lambda_Bulan(t) - lambda_Matahari(t)
        delta_raw = moon_ecl[i] - sun_ecl[i]
        out.append({
            "jd": date_to_jd(ep), "eclMoonDeg": moon_ecl[i], "eclSunDeg": sun_ecl[i],
            # --- PERSAMAAN (3.2) - FUNGSI TARGET f(t) = wrapTo180(d-lambda)
            "deltaRawDeg": delta_raw, "fDeg": wrap_to_180(delta_raw),
        })
    return out


def _eval_f(t: datetime) -> float:
    return _eval_f_batch_detailed([t])[0]["fDeg"]


# ======================================================================
# PERSAMAAN (3.7) - TURUNAN NUMERIK (CENTRAL DIFFERENCE)
# LANGKAH 9 dari 14 -> f'(tn) = [ f(tn+d) - f(tn-d) ] / 2d , d = 60 detik
# Turunan TIDAK dihitung analitik, tetapi didekati dari 3 titik waktu.
# ======================================================================
def _eval_f_and_prime(t: datetime) -> Tuple[float, float]:
    # 3 titik waktu: tn-60 detik, tn, tn+60 detik
    t_minus = t - timedelta(seconds=DELTA_S)
    t_plus = t + timedelta(seconds=DELTA_S)
    d_minus, d_mid, d_plus = _eval_f_batch_detailed([t_minus, t, t_plus])
    # --- rumus beda-tengah: selisih f dibagi 2*delta
    f_prime = (d_plus["fDeg"] - d_minus["fDeg"]) / (2 * DELTA_S)
    return d_mid["fDeg"], f_prime


# ======================================================================
# PEMINDAIAN AWAL (BRACKETING) - LANGKAH 2, 6, 7 dari 14
# Pindai seluruh rentang tiap 6 jam -> cari perubahan tanda f(t)
# -> saring oposisi/purnama -> bentuk bracket konjungsi
# ======================================================================
def scan_for_brackets(start_utc: datetime, end_utc: datetime) -> List[Dict]:
    """6-hour scan for sign changes in f(t); opposition brackets filtered out.
    Port of newMoonNR.ts:scanForBrackets. Batches that still fail after
    MAX_RETRIES (Horizons persistently down/rate-limiting) are SKIPPED rather
    than aborting the whole scan — see _run_concurrent_tolerant in Section 4.
    A skipped batch means the corresponding ~10-day span (BATCH_SIZE x 6h) has
    no scan points, so a sign change could theoretically be missed there; the
    successfully-scanned batches are unaffected and (via the disk cache)
    re-running this cell only retries the batches that failed, so coverage
    converges to complete over a few attempts instead of starting from zero."""
    # --- LANGKAH 2 - SUSUN DAFTAR EPOCH PEMINDAIAN AWAL (tiap 6 jam) ----
    #     2017-2026 (3.652 hari) x 4 titik/hari = 14.608 epoch
    scan_epochs = []
    t = start_utc
    while t <= end_utc:
        scan_epochs.append(t)
        t += SCAN_STEP
    _conjunction_scan_stats["totalEpochs"] += len(scan_epochs)  # Tabel 4.4: jumlah epoch pemindaian awal

    batches = [scan_epochs[i:i + BATCH_SIZE] for i in range(0, len(scan_epochs), BATCH_SIZE)]
    log_step(f"Memindai {len(scan_epochs)} titik waktu (langkah 6 jam) dalam {len(batches)} batch Horizons...")
    detail_batches, failures = _run_concurrent_tolerant(
        [lambda b=b: _eval_f_batch_detailed(b) for b in batches], progress_label="Pemindaian batch")

    all_detail: List[Dict] = []
    kept_epochs: List[datetime] = []
    for bi, batch in enumerate(batches):
        if detail_batches[bi] is not None:
            all_detail.extend(detail_batches[bi])
            kept_epochs.extend(batch)
    _conjunction_scan_stats["totalRawValues"] += len(all_detail) * 2  # Tabel 4.4: nilai Bulan + Matahari

    if failures:
        log_warn(f"{len(failures)}/{len(batches)} batch GAGAL dipindai setelah {MAX_RETRIES}x percobaan "
                 "(Horizons kemungkinan sedang overload/membatasi trafik) — titik waktu pada batch "
                 "tsb dilewati untuk sementara. JALANKAN ULANG sel ini agar melengkapi cakupan; "
                 "batch yang sudah berhasil tidak akan di-query ulang (sudah tersimpan di cache).")

    all_f = [d["fDeg"] for d in all_detail]
    brackets = []
    for i in range(1, len(all_f)):
        # --- LANGKAH 6 | PERSAMAAN (3.3) - DETEKSI PERUBAHAN TANDA ------
        #     sign change terjadi bila f(t1) x f(t2) < 0  (247 terdeteksi)
        if all_f[i - 1] * all_f[i] < 0:
            _conjunction_scan_stats["totalSignChanges"] += 1  # Tabel 4.6: total deteksi sign change
            # --- PENYARINGAN OPOSISI / PURNAMA (bukan konjungsi) --------
            #     124 perubahan tanda disaring di sini
            if abs(all_f[i - 1]) > 90 or abs(all_f[i]) > 90:
                _conjunction_scan_stats["filteredAsOpposition"] += 1  # Tabel 4.6: disaring sbg oposisi
                continue  # opposition (full moon), not conjunction
            # --- LANGKAH 7 | PERSAMAAN (3.4) - BENTUK BRACKET -----------
            #     B = [t1, t2] -> tersisa 123 bracket konjungsi valid
            brackets.append({"t1": kept_epochs[i - 1], "t2": kept_epochs[i],
                              "f1": all_f[i - 1], "f2": all_f[i]})
    _conjunction_scan_stats["bracketsFormed"] += len(brackets)  # Tabel 4.6: bracket konjungsi diproses
    coverage_note = f" (CAKUPAN TIDAK LENGKAP: {len(failures)} batch gagal)" if failures else ""
    log_ok(f"Pemindaian selesai — {len(brackets)} bracket (kandidat perubahan tanda f(t)) ditemukan{coverage_note}")
    return brackets


# ======================================================================
# MEKANISME CADANGAN - FALLBACK BISECTION
# Dipakai HANYA bila Newton-Raphson gagal konvergen dalam 30 iterasi.
# Hasil periode uji 2017-2026: TIDAK PERNAH TERPAKAI (0 dari 123).
# ======================================================================
def bisection(t1: datetime, t2: datetime, f1: float) -> Tuple[datetime, float]:
    """Refine bracket to ~2s precision. Port of newMoonNR.ts:bisection."""
    lo, hi = t1.timestamp(), t2.timestamp()
    f_lo = f1
    while hi - lo > 2:
        mid = (lo + hi) / 2
        f_mid = _eval_f(datetime.fromtimestamp(mid, tz=timezone.utc))
        if f_lo * f_mid < 0:
            hi = mid
        else:
            lo, f_lo = mid, f_mid
    t_mid = (lo + hi) / 2
    return datetime.fromtimestamp(t_mid, tz=timezone.utc), hi - lo


# ======================================================================
# ITERASI NEWTON-RAPHSON PADA SATU BRACKET
# LANGKAH 8, 10, 11, 12 dari 14
# Persamaan (3.5) tebakan awal -> (3.8) koreksi waktu -> (3.6) update
# ======================================================================
def try_nr_on_bracket(bracket: Dict, window_start: datetime, window_end: datetime) -> Optional[Dict]:
    """Newton-Raphson on a single bracket, falling back to bisection if it does
    not converge within MAX_ITER. Port of newMoonNR.ts:tryNROnBracket."""
    # --- LANGKAH 8 | PERSAMAAN (3.5) - TEBAKAN AWAL (INITIAL GUESS) ----
    #     t0 = titik tengah bracket = (t1 + t2) / 2
    t = bracket["t1"] + (bracket["t2"] - bracket["t1"]) / 2
    iterations = []
    converged = False

    # --- LANGKAH 12 - ULANGI ITERASI SAMPAI TOLERANSI TERPENUHI --------
    for i in range(MAX_ITER):
        # nilai fungsi f(tn) dan turunan f'(tn) pada tebakan saat ini
        f, f_prime = _eval_f_and_prime(t)
        # --- LANGKAH 10 | PERSAMAAN (3.8) - KOREKSI WAKTU (stepSec) ----
        #     stepSec = -f(tn) / f'(tn)
        step_sec = 3600.0 if abs(f_prime) < 1e-15 else -(f / f_prime)

        # --- KRITERIA KONVERGENSI (dua batas toleransi) ----------------
        #     (a) RESIDUAL ABSOLUT: |f(tn)| <= 1e-6 derajat
        converged_by_angle = abs(f) < EPS_ANGLE
        #     (b) ABSOLUTE APPROXIMATE ERROR: |stepSec| <= 0,2 detik
        #         (dengan residual masih dalam batas kecil, < 0,01 derajat)
        converged_by_time_and_angle = abs(step_sec) < EPS_TIME and abs(f) < 0.01
        converged_this_step = converged_by_angle or converged_by_time_and_angle

        iterations.append({"iteration": i + 1, "t": t, "fDeg": f, "fPrime": f_prime,
                            "stepSec": step_sec, "converged": converged_this_step})

        if converged_this_step:
            converged = True
            break

        # --- LANGKAH 11 | PERSAMAAN (3.6) - PERBARUI WAKTU TEBAKAN -----
        #     t(n+1) = tn + stepSec   [ = tn - f(tn)/f'(tn) ]
        t = t + timedelta(seconds=step_sec)
        if t < window_start:
            t = window_start + timedelta(hours=1)
        if t > window_end:
            t = window_end - timedelta(hours=1)

    if not converged:
        try:
            t_bis, _delta_sec = bisection(bracket["t1"], bracket["t2"], bracket["f1"])
            f_bis = _eval_f(t_bis)
            if abs(f_bis) < 1:
                iterations.append({"iteration": len(iterations) + 1, "t": t_bis, "fDeg": f_bis,
                                    "fPrime": 0, "stepSec": 0, "converged": True, "bisection": True})
                _conjunction_scan_stats["totalNRIterations"] += len(iterations)  # Tabel 4.6: total iterasi
                _conjunction_scan_stats["bracketsConverged"] += 1
                _conjunction_scan_stats["bracketsFallback"] += 1  # Tabel 4.6: butuh fallback bisection
                return {"t": t_bis, "iterations": iterations, "converged": True, "usedBisection": True}
            _conjunction_scan_stats["bracketsFallback"] += 1  # fallback dicoba tapi tetap gagal
            return None
        except Exception:
            return None

    _conjunction_scan_stats["totalNRIterations"] += len(iterations)  # Tabel 4.6: total iterasi Newton-Raphson
    _conjunction_scan_stats["bracketsConverged"] += 1  # Tabel 4.6: bracket konvergen tanpa fallback
    return {"t": t, "iterations": iterations, "converged": True, "usedBisection": False}


# ======================================================================
# LANGKAH 13 - TETAPKAN WAKTU KONJUNGSI (KELUARAN TAHAP MODELING)
# Menjalankan seluruh alur: scan -> bracket -> Newton-Raphson.
# Hasil periode 2017-2026: 123 waktu konjungsi valid.
# ======================================================================
def find_conjunctions_in_range(start_utc: datetime, end_utc: datetime) -> List[ConjSimple]:
    """Find ALL conjunctions in [start_utc, end_utc], sorted ascending.
    Port of newMoonNR.ts:findConjunctionsInRange. Brackets are refined
    concurrently (independent, no shared state) exactly like Promise.all() in
    production — see _run_concurrent_tolerant in Section 4. A bracket whose
    refinement queries keep failing after MAX_RETRIES is skipped (logged, not
    fatal) rather than aborting every other (already-succeeding) bracket."""
    brackets = scan_for_brackets(start_utc, end_utc)
    if not brackets:
        log_warn("Tidak ada bracket konjungsi ditemukan pada rentang ini.")
        return []
    log_step(f"Menjalankan Newton-Raphson pada {len(brackets)} bracket...")
    nr_results, failures = _run_concurrent_tolerant(
        [lambda b=b: try_nr_on_bracket(b, start_utc, end_utc) for b in brackets],
        progress_label="Refinement Newton-Raphson")
    if failures:
        log_warn(f"{len(failures)}/{len(brackets)} bracket GAGAL di-refine (Horizons bermasalah) — "
                 "dilewati untuk sementara. Jalankan ulang sel ini untuk melengkapi.")
    results = []
    for nr in nr_results:
        if nr and nr["converged"]:
            results.append(ConjSimple(t=nr["t"], iso=_iso_z(nr["t"]), converged=True))
    results.sort(key=lambda c: c.t)
    coverage_note = f" (CAKUPAN TIDAK LENGKAP: {len(failures)} bracket gagal)" if failures else ""
    log_ok(f"Newton-Raphson selesai — {len(results)}/{len(brackets)} bracket konvergen menjadi konjungsi{coverage_note}")
    return results


# ======================================================================
# DEDUPLIKASI HASIL KONJUNGSI - AMBANG 12 JAM
# Membuang hasil kembar dari bracket bertetangga yang konvergen ke
# peristiwa yang sama. Hasil periode uji: 0 duplikat.
# ======================================================================
def dedup_conjunctions(conjunctions: List[ConjSimple]) -> List[ConjSimple]:
    """Remove near-duplicate conjunctions from adjacent scan brackets converging
    to the same event (<=12h apart). Port of the dedup step in
    api/konjungsi-periode/route.ts."""
    out = []
    for i, c in enumerate(conjunctions):
        if i == 0:
            out.append(c)
            continue
        if (c.t - conjunctions[i - 1].t) > DEDUP_THRESHOLD:
            out.append(c)
    return out


def find_conjunction(window_start: datetime, window_end: datetime) -> Dict:
    """Single best (earliest converged) conjunction in a window — used by the
    local Wujudul Hilal pipeline (Section 12). Port of newMoonNR.ts:findConjunction."""
    brackets = scan_for_brackets(window_start, window_end)
    if not brackets:
        raise RuntimeError(f"No conjunction found in window {window_start.isoformat()} to {window_end.isoformat()}")

    sorted_brackets = sorted(brackets, key=lambda b: b["t1"] + (b["t2"] - b["t1"]) / 2)
    nr_results, failures = _run_concurrent_tolerant(
        [lambda b=b: try_nr_on_bracket(b, window_start, window_end) for b in sorted_brackets])
    if failures:
        log_warn(f"{len(failures)}/{len(sorted_brackets)} bracket gagal di-refine pada jendela "
                 f"{window_start.date()}..{window_end.date()} — dilewati untuk pemilihan konjungsi ini.")
    converged = [{**r, "bracket": b} for r, b in zip(nr_results, sorted_brackets) if r and r["converged"]]

    if converged:
        converged.sort(key=lambda r: r["t"])
        best = converged[0]
    else:
        fb = sorted_brackets[0]
        t0 = fb["t1"] + (fb["t2"] - fb["t1"]) / 2
        best = {"t": t0, "iterations": [], "converged": False, "bracket": fb}

    t, bracket = best["t"], best["bracket"]
    bisection_delta_sec = None
    try:
        t_bis, _ = bisection(bracket["t1"], bracket["t2"], bracket["f1"])
        bisection_delta_sec = abs((t - t_bis).total_seconds())
    except Exception:
        pass

    if best["converged"]:
        log_ok(f"Konjungsi ditemukan: {_iso_z(t)}")
    else:
        log_fail(f"Tidak konvergen pada jendela {window_start.isoformat()}..{window_end.isoformat()} — memakai estimasi bracket kasar")

    return {
        "conjunctionUTC": t, "conjunctionISO": _iso_z(t), "converged": best["converged"],
        "totalIterations": len(best["iterations"]), "bisectionDeltaSec": bisection_delta_sec,
        "bisectionWarning": bisection_delta_sec is not None and bisection_delta_sec > 2,
        "iterations": best["iterations"],
    }


## 9. Aturan Evaluasi: Rule A/Rule B (Wujudul Hilal) & Ambang KHGT

Porting dari `src/lib/wujudulHilalRule.ts` dan `src/lib/khgtRule.ts`.

- **Rule A** — konjungsi terjadi sebelum waktu Matahari terbenam pada tanggal evaluasi D.
- **Rule B** — ketinggian (altitude) topocentris Bulan saat Matahari terbenam D lebih besar dari 0°.
- Jika Rule A dan Rule B terpenuhi pada D → **1 Ramadan = D+1**; jika belum, evaluasi berlanjut
  ke D+1, D+2, D+3 (istikmal maksimum).
- **KHGT** (informasi pembanding global, bukan fokus utama): lolos jika altitude geosentris
  Bulan ≥ 5° **dan** elongasi geosentris Bulan-Matahari ≥ 8°.

In [ ]:
# ======================================================================
# LANGKAH 14 dari 14 - EVALUASI PASCA-KONJUNGSI (TAHAP EVALUATION)
# PERSAMAAN (3.9) RULE A + PERSAMAAN (3.10) RULE B
# Dievaluasi di lokasi pengamatan utama: Kota Bekasi, Jawa Barat.
# ======================================================================
def check_wujudul_hilal(conjunction_utc: datetime, sunset_utc: datetime,
                         moon_alt_at_sunset_deg: float, candidate_date: str) -> Dict:
    """Rule A / Rule B (Muhammadiyah-style Wujudul Hilal criterion).
    Port of wujudulHilalRule.ts:checkWujudulHilal."""
    # --- PERSAMAAN (3.9) - RULE A --------------------------------------
    #     konjungsi terjadi SEBELUM Matahari terbenam: t_konjungsi < t_sunset
    rule_a = conjunction_utc < sunset_utc
    # --- PERSAMAAN (3.10) - RULE B -------------------------------------
    #     ketinggian (altitude) topocentric Bulan saat sunset > 0 derajat
    rule_b = moon_alt_at_sunset_deg > 0
    # --- Kedua aturan harus terpenuhi bersamaan ------------------------
    #     Jika terpenuhi pada sunset tanggal D -> 1 Ramadan = D+1
    fulfilled = rule_a and rule_b
    # --- Penanda kasus borderline (altitude sangat tipis, mis. 2024) ---
    is_borderline = abs(moon_alt_at_sunset_deg) <= 0.2
    return {
        "ruleA": rule_a, "ruleB": rule_b, "fulfilled": fulfilled,
        "candidateDate": candidate_date, "moonAltAtSunsetDeg": moon_alt_at_sunset_deg,
        "isBorderline": is_borderline,
    }


KHGT_ALT_THRESHOLD = 5.0    # degrees
KHGT_ELONG_THRESHOLD = 8.0  # degrees


def check_khgt(geo_alt_deg: float, geo_elong_deg: float) -> Dict:
    """KHGT threshold check on geocentric Moon altitude & elongation.
    Port of khgtRule.ts:checkKHGT."""
    alt_margin = geo_alt_deg - KHGT_ALT_THRESHOLD
    elong_margin = geo_elong_deg - KHGT_ELONG_THRESHOLD
    return {
        "pass": alt_margin >= 0 and elong_margin >= 0,
        "geoAltDeg": geo_alt_deg, "geoElongDeg": geo_elong_deg,
        "altMargin": alt_margin, "elongMargin": elong_margin,
    }


## 10. Estimator Tanggal Konjungsi (untuk klasifikasi kandidat & jendela pencarian)

Dua estimator aritmetika independen dipakai di aplikasi produksi (dipertahankan terpisah,
sesuai kode asli, walau formulanya mirip):

- `estimate_ramadan_conj_date(year)` — dipakai Fase 1 & pipeline KHGT global untuk menentukan
  konjungsi mana yang menjadi "kandidat awal Ramadan" per tahun Masehi (`khgtPipeline.ts`).
- `estimate_ramadan1(year)` — dipakai pipeline lokal Wujudul Hilal sebagai jendela pencarian
  awal (`ramadanFromSyaban.ts`). Versi produksinya memiliki jangkar (*anchor*) opsional dari
  `data/anchors_syaban.json`; **file anchor ini tidak tersedia** di proyek ini (dihapus dari
  repo), sehingga baik produksi maupun notebook ini **selalu memakai jalur estimator inline**
  di bawah — ini bukan penyederhanaan, melainkan perilaku produksi yang sebenarnya saat ini.

Kedua estimator memakai teknik "koreksi tahun Hijriah nyata" yang sama: estimasi linear dari
titik acuan, lalu digeser maju/mundur dalam kelipatan panjang tahun Hijriah (354,36667 hari)
sampai tahun Masehi hasilnya berada pada `[year-1, year]` — supaya tidak melenceng untuk tahun
yang jauh dari titik acuan (dikonfirmasi valid sampai tahun 2090 pada kode produksi).

In [ ]:
ISLAMIC_YEAR_DAYS = 354.36667


def estimate_ramadan_conj_date(year: int) -> datetime:
    """Day-level estimate of the Ramadan conjunction date for a Gregorian year.
    Reference: Ramadan 2024 conjunction ~ 2024-03-10.
    Port of khgtPipeline.ts:estimateRamadanConjDate."""
    ref_conj = datetime(2024, 3, 10, 12, 0, 0, tzinfo=timezone.utc)
    year_diff = year - 2024
    est = ref_conj + timedelta(days=year_diff * ISLAMIC_YEAR_DAYS)
    while est.year < year - 1:
        est += timedelta(days=round(ISLAMIC_YEAR_DAYS))
    while est.year > year:
        est -= timedelta(days=round(ISLAMIC_YEAR_DAYS))
    return est


def estimate_ramadan1(year: int) -> datetime:
    """Inline fallback estimator for 1 Ramadan (no anchor file needed).
    Reference: 1 Ramadan 2025 ~ 2025-03-01.
    Port of ramadanFromSyaban.ts:estimateRamadan1."""
    seed_ramadan1 = datetime(2025, 3, 1, tzinfo=timezone.utc)
    year_diff = year - 2025
    est = seed_ramadan1 + timedelta(days=year_diff * ISLAMIC_YEAR_DAYS)
    while est.year < year - 1:
        est += timedelta(days=round(ISLAMIC_YEAR_DAYS))
    while est.year > year:
        est -= timedelta(days=round(ISLAMIC_YEAR_DAYS))
    return est


## 11. FASE 1 — Evaluasi Konjungsi

Porting dari `src/app/api/konjungsi-periode/route.ts`. Dua tahap, sesuai arsitektur
two-phase aplikasi asli:

- **`run_phase1`** (cepat): memindai SEMUA konjungsi nyata pada rentang tahun, menghilangkan
  duplikat (<12 jam), mengklasifikasikan konjungsi mana yang menjadi kandidat awal Ramadan
  tiap tahun (konjungsi TERDEKAT dengan `estimate_ramadan_conj_date(year)` — **tanpa** ambang
  batas hari, persis seperti produksi), dan menghitung waktu Matahari terbenam + umur Bulan
  saat itu untuk setiap konjungsi.
- **`enrich_phase1_candidates`** (setara tombol "Muat Parameter Detail" pada aplikasi web):
  untuk baris yang berstatus kandidat saja, menambahkan bujur ekliptika Bulan/Matahari
  (validitas: `|selisih| <= 2°` saat konjungsi), elongasi & altitude geosentris, RA/Dec,
  azimuth/altitude topocentris, status KHGT (info pembanding), dan verdict Rule A/Rule B.

In [ ]:
def run_phase1(from_year: int, to_year: int, lat: float, lon: float, tz: str):
    """Port of /api/konjungsi-periode Phase 1. Returns (DataFrame, state) —
    `state` carries the raw conjunction objects + candidate flags needed by
    enrich_phase1_candidates() below."""
    log_step(f"Mengatur periode tahun {from_year}-{to_year}")
    window_start = datetime(from_year, 1, 1, tzinfo=timezone.utc)
    window_end = datetime(to_year, 12, 31, 23, 59, 59, tzinfo=timezone.utc)
    log_ok(f"Periode tahun {from_year}-{to_year} berhasil diatur ({window_start.date()} s.d. {window_end.date()})")

    log_step("Memindai & menemukan seluruh konjungsi (Newton-Raphson)...")
    raw = find_conjunctions_in_range(window_start, window_end)
    log_ok(f"Ditemukan {len(raw)} konjungsi mentah")

    log_step("Menghapus konjungsi duplikat (jarak < 12 jam dari bracket scan bertetangga)...")
    conjunctions = dedup_conjunctions(raw)
    log_ok(f"Deduplikasi berhasil — {len(conjunctions)} konjungsi unik tersisa")

    log_step(f"Mengklasifikasikan kandidat awal Ramadan untuk {to_year - from_year + 1} tahun ({from_year}-{to_year})...")
    candidate_iso: Dict[str, bool] = {}
    candidate_note: Dict[str, str] = {}
    candidate_dist: Dict[str, float] = {}
    for year in range(from_year, to_year + 1):
        est = estimate_ramadan_conj_date(year)
        best_idx, best_dist = -1, float("inf")
        for i, c in enumerate(conjunctions):
            dist = abs((c.t - est).total_seconds())
            if dist < best_dist:
                best_dist, best_idx = dist, i
        if best_idx < 0:
            log_fail(f"Tahun {year}: tidak ada konjungsi ditemukan di rentang hasil pemindaian")
            continue
        iso = conjunctions[best_idx].iso
        dist_days = round(best_dist / 86400, 1)
        est_str = est.strftime("%Y-%m-%d")
        if dist_days <= 15:
            note = f"Kandidat awal Ramadan untuk tahun target {year} — selisih {dist_days} hari dari estimasi ({est_str})"
        elif dist_days <= 35:
            note = (f"Kandidat awal Ramadan untuk tahun target {year} — selisih {dist_days} hari dari estimasi. "
                    "Periksa kelengkapan data HORIZONS untuk tahun ini.")
        else:
            note = (f"Kandidat perkiraan Ramadan tahun {year} — selisih {dist_days} hari dari estimasi. "
                    "Data konjungsi kemungkinan tidak lengkap untuk tahun ini.")
        candidate_iso[iso] = True
        candidate_note[iso] = note
        candidate_dist[iso] = dist_days
        log_ok(f"Tahun {year}: kandidat {conjunctions[best_idx].t.strftime('%Y-%m-%d')} (selisih {dist_days} hari)")
    log_ok(f"Klasifikasi kandidat selesai — {len(candidate_iso)}/{to_year - from_year + 1} tahun mendapat kandidat")

    log_step(f"Menghitung waktu Matahari terbenam & umur Bulan untuk {len(conjunctions)} konjungsi (lat={lat}, lon={lon}, tz={tz})...")
    rows, sunset_cache = [], {}
    for c in conjunctions:
        date_str = c.t.strftime("%Y-%m-%d")
        try:
            ss = get_sunset(date_str, lat, lon, tz)
        except Exception:
            ss = None
        sunset_cache[c.iso] = ss

        sunset_utc = ss["sunsetUTC"] if ss else None
        moon_age_hours = round((sunset_utc - c.t).total_seconds() / 3600, 3) if sunset_utc else None
        is_candidate = c.iso in candidate_iso

        rows.append({
            "year": c.t.year, "conjDate": date_str, "conjTimeUTC": c.t.strftime("%H:%M:%S"),
            "conjISO": c.iso,
            "sunsetLocal": ss["sunsetLocal"] if ss else None,
            "sunsetUTC": sunset_utc.isoformat() if sunset_utc else None,
            "moonAgeHours": moon_age_hours,
            "isRamadanCandidate": is_candidate,
            "candidateNote": candidate_note.get(c.iso, "Bukan kandidat awal Ramadan") if is_candidate else "Bukan kandidat awal Ramadan",
            "candidateDistDays": candidate_dist.get(c.iso) if is_candidate else None,
        })
    log_ok(f"Perhitungan Matahari terbenam & umur Bulan selesai untuk {len(rows)} konjungsi")

    df = pd.DataFrame(rows)
    state = {"conjunctions": conjunctions, "candidateIso": candidate_iso, "sunsetCache": sunset_cache,
             "lat": lat, "lon": lon, "tz": tz, "baseDf": df}
    return df, state


def enrich_phase1_candidates(state: Dict, lat: Optional[float] = None, lon: Optional[float] = None,
                              tz: Optional[str] = None) -> pd.DataFrame:
    """Port of /api/konjungsi-periode Phase 2 enrichment (scope='candidates') —
    equivalent to clicking "Muat Parameter Detail" in the web app. Queries are
    issued per-candidate-row here (candidates are typically ~10-13 rows for a
    10-year range) rather than production's single batched request across all
    candidates — a readability simplification that does not change any result."""
    lat = lat if lat is not None else state["lat"]
    lon = lon if lon is not None else state["lon"]
    tz = tz if tz is not None else state["tz"]
    conjunctions: List[ConjSimple] = state["conjunctions"]
    candidate_iso: Dict[str, bool] = state["candidateIso"]
    sunset_cache = state["sunsetCache"]

    detail_cols = ["eclMoonDeg", "eclSunDeg", "eclDiffDeg", "eclDataValid", "geoElongDeg", "geoMoonAltDeg",
                    "raMoonDeg", "decMoonDeg", "raSunDeg", "decSunDeg", "topoMoonAltDeg", "topoMoonAzDeg",
                    "khgtPass", "khgtAltMargin", "khgtElongMargin",
                    "whRuleA", "whRuleB", "whFulfilled", "whIsBorderline", "whMoonAltAtSunsetDeg"]

    n_candidates = sum(1 for c in conjunctions if c.iso in candidate_iso)
    log_step(f"Memuat parameter detail untuk {n_candidates} baris kandidat (lat={lat}, lon={lon}, tz={tz})...")
    rows = []
    for c in conjunctions:
        is_candidate = c.iso in candidate_iso
        row = {"conjISO": c.iso}
        if not is_candidate:
            row.update({k: None for k in detail_cols})
            rows.append(row)
            continue

        log_step(f"  Konjungsi {c.t.strftime('%Y-%m-%d')}: bujur ekliptika, geosentris, topocentris, KHGT, Rule A/B...")
        ss = sunset_cache.get(c.iso)
        ecl_moon = get_ecliptic_lon(MOON_CMD, [c.t])[0]
        ecl_sun = get_ecliptic_lon(SUN_CMD, [c.t])[0]
        ecl_diff = round(ecl_moon - ecl_sun, 6)
        ecl_valid = abs(ecl_diff) <= 2.0

        ra_dec_m = get_geocentric_apparent_radec(MOON_CMD, [ss["sunsetUTC"]])[0] if ss else None
        ra_dec_s = get_geocentric_apparent_radec(SUN_CMD, [ss["sunsetUTC"]])[0] if ss else None
        if ra_dec_m and ra_dec_s and ss:
            geo_elong = round(geocentric_elong_deg(ra_dec_m[0], ra_dec_m[1], ra_dec_s[0], ra_dec_s[1]), 4)
            geo_alt = round(geocentric_alt_deg(ra_dec_m[0], ra_dec_m[1], lat, lon, ss["sunsetUTC"]), 4)
        else:
            geo_elong = geo_alt = None

        topo = get_topo_azel(MOON_CMD, [ss["sunsetUTC"]], lat, lon)[0] if ss else None
        khgt = check_khgt(geo_alt, geo_elong) if (geo_alt is not None and geo_elong is not None) else None

        wh = None
        if topo and ss:
            candidate_date = ss["sunsetLocal"][:10] if ss["sunsetLocal"] else c.t.strftime("%Y-%m-%d")
            wh = check_wujudul_hilal(c.t, ss["sunsetUTC"], topo[1], candidate_date)

        row.update({
            "eclMoonDeg": round(ecl_moon, 6), "eclSunDeg": round(ecl_sun, 6), "eclDiffDeg": ecl_diff,
            "eclDataValid": ecl_valid, "geoElongDeg": geo_elong, "geoMoonAltDeg": geo_alt,
            "raMoonDeg": round(ra_dec_m[0], 4) if ra_dec_m else None,
            "decMoonDeg": round(ra_dec_m[1], 4) if ra_dec_m else None,
            "raSunDeg": round(ra_dec_s[0], 4) if ra_dec_s else None,
            "decSunDeg": round(ra_dec_s[1], 4) if ra_dec_s else None,
            "topoMoonAltDeg": round(topo[1], 4) if topo else None,
            "topoMoonAzDeg": round(topo[0], 4) if topo else None,
            "khgtPass": khgt["pass"] if khgt else None,
            "khgtAltMargin": round(khgt["altMargin"], 4) if khgt else None,
            "khgtElongMargin": round(khgt["elongMargin"], 4) if khgt else None,
            "whRuleA": wh["ruleA"] if wh else None, "whRuleB": wh["ruleB"] if wh else None,
            "whFulfilled": wh["fulfilled"] if wh else None, "whIsBorderline": wh["isBorderline"] if wh else None,
            "whMoonAltAtSunsetDeg": round(wh["moonAltAtSunsetDeg"], 4) if wh else None,
        })
        rows.append(row)
        verdict = "Rule A & B terpenuhi" if (wh and wh["fulfilled"]) else "Rule A/B belum terpenuhi"
        log_ok(f"  Konjungsi {c.t.strftime('%Y-%m-%d')} selesai — {verdict}")

    detail_df = pd.DataFrame(rows)
    log_ok(f"Parameter detail selesai dimuat untuk {n_candidates} baris kandidat")
    return state["baseDf"].merge(detail_df, on="conjISO", how="left")


### 11.1 Jalankan Fase 1

> Disarankan uji coba dulu dengan rentang pendek (mis. `FROM_YEAR = TO_YEAR = 2024`) di
> Bagian 1 sebelum menjalankan rentang penuh 2017–2026 (lihat "Catatan Penting" di atas).

In [ ]:
log_step(f"===== FASE 1: Evaluasi Konjungsi {FROM_YEAR}-{TO_YEAR} =====")
t0 = time.time()
df_phase1, phase1_state = run_phase1(FROM_YEAR, TO_YEAR, LAT, LON, TZ)
log_ok(f"FASE 1 selesai dalam {time.time() - t0:.1f} detik — total {len(df_phase1)} konjungsi ditemukan")
df_phase1


In [ ]:
# Ringkasan kandidat awal Ramadan per tahun (hasil klasifikasi Fase 1)
df_phase1[df_phase1["isRamadanCandidate"]][
    ["year", "conjDate", "conjTimeUTC", "sunsetLocal", "moonAgeHours", "candidateNote"]
].reset_index(drop=True)


### 11.2 Muat Parameter Detail (setara tombol Fase 2 di halaman Evaluasi Konjungsi)

In [ ]:
log_step("===== FASE 1.2: Muat Parameter Detail =====")
t0 = time.time()
df_phase1_detail = enrich_phase1_candidates(phase1_state)
log_ok(f"FASE 1.2 selesai dalam {time.time() - t0:.1f} detik")
df_phase1_detail[df_phase1_detail["isRamadanCandidate"]].reset_index(drop=True)


### 11.3 Lampiran: Statistik Tahap Data Preparation & Modeling (Tabel 4.4 & 4.6)

Bagian ini **baru ditambahkan** untuk menjawab pertanyaan "angka-angka di Bab IV didapat dari
mana?" pada tahap Data Preparation dan Modeling -- sebelum masuk ke contoh iterasi Newton-Raphson
detail di Bagian 18.

- **`tampilkan_statistik_data_preparation_modeling()`** membaca `_conjunction_scan_stats`, yaitu
  penghitung yang **sudah otomatis terisi** saat Bagian 11.1 (Fase 1) dijalankan -- `scan_for_brackets`
  dan `try_nr_on_bracket` (Bagian 8) masing-masing menambah 1 baris kode pencatat statistik
  (jumlah epoch, total sign change, jumlah disaring sebagai oposisi, jumlah bracket, total iterasi
  NR) **tanpa mengubah nilai balik atau alur fungsi tersebut sama sekali** -- Fase 1 & Fase 2 tetap
  menghasilkan output yang identik seperti sebelumnya. Fungsi ini **tidak melakukan query baru ke
  Horizons** -- hanya membaca angka yang sudah terkumpul dari scan yang sama dengan Fase 1.
- **`contoh_transformasi_wrapto180(...)`** mengambil beberapa sampel epoch (default: 10 epoch,
  berjarak 15 hari, dimulai 1 Januari `FROM_YEAR`) dan menampilkan bujur ekliptika mentah, selisih
  `Δλ` (Persamaan 3.1), dan hasil `wrapTo180` (Persamaan 3.2) -- persis format Tabel 4.5.

**Bagian ini sudah ditempatkan tepat setelah Fase 1** supaya urutan "Run All" dari atas ke
bawah otomatis menghasilkan angka yang benar. **Perhatian:** `_conjunction_scan_stats` adalah
penghitung global yang terus bertambah setiap kali `scan_for_brackets`/`try_nr_on_bracket`
dipanggil di sel manapun sesudah ini -- termasuk Fase 2 (Bagian 15-16, yang juga memindai
konjungsi untuk KHGT & Wujudul Hilal lokal) dan demo Bagian 18. Jadi:
- Lihat tabel di Bagian 11.3.1 **sebelum** menjalankan Fase 2, supaya angkanya murni dari
  pemindaian 2017-2026 di Fase 1 saja (sesuai Tabel 4.4/4.6 skripsi).
- Kalau kamu sudah terlanjur menjalankan Fase 2 (atau ingin melihat tabel ini lagi setelahnya),
  panggil `reset_conjunction_scan_stats()` lalu jalankan ulang Bagian 11.1 sebelum membaca ulang
  tabel di Bagian 11.3.1.

In [ ]:
def tampilkan_statistik_data_preparation_modeling() -> pd.DataFrame:
    """Reproduksi Tabel 4.4 (Data Preparation) & Tabel 4.6 (Modeling) dari
    _conjunction_scan_stats -- dicatat sebagai efek-samping oleh scan_for_brackets
    dan try_nr_on_bracket (Bagian 8) saat Fase 1 (Bagian 11.1) dijalankan. Tidak
    melakukan query Horizons baru; murni membaca angka yang sudah terkumpul."""
    s = _conjunction_scan_stats
    avg_iter = round(s["totalNRIterations"] / s["bracketsConverged"], 2) if s["bracketsConverged"] else None
    rows = [
        ("Jumlah epoch pemindaian awal", s["totalEpochs"], "Tabel 4.4 -- titik waktu langkah 6 jam"),
        ("Total nilai bujur ekliptika mentah", s["totalRawValues"], "Tabel 4.4 -- Bulan + Matahari"),
        ("Total deteksi sign change", s["totalSignChanges"], "Tabel 4.6 -- perubahan tanda f(t)"),
        ("Disaring sebagai oposisi/purnama", s["filteredAsOpposition"], "Tabel 4.6 -- bukan konjungsi"),
        ("Bracket konjungsi diproses", s["bracketsFormed"], "Tabel 4.6 -- diteruskan ke Newton-Raphson"),
        ("Total iterasi Newton-Raphson", s["totalNRIterations"], "Tabel 4.6 -- seluruh bracket"),
        ("Bracket konvergen", s["bracketsConverged"], "Tabel 4.6/4.9 -- berhasil diselesaikan NR"),
        ("Rata-rata iterasi per konjungsi", avg_iter, "totalNRIterations / bracketsConverged"),
        ("Bracket butuh fallback bisection", s["bracketsFallback"], "Tabel 4.6/4.9"),
    ]
    return pd.DataFrame(rows, columns=["Komponen", "Nilai", "Keterangan"])


def contoh_transformasi_wrapto180(from_year: int = None, n_sample: int = 10,
                                   spacing_hari: int = 15) -> pd.DataFrame:
    """Reproduksi Tabel 4.5: sampel epoch pemindaian awal beserta selisih bujur
    ekliptika mentah (Persamaan 3.1) dan hasil wrapTo180 (Persamaan 3.2). Hanya
    memanggil get_ecliptic_lon & wrap_to_180 (sudah ada di Bagian 5 & 2) --
    tidak menyentuh scan_for_brackets/run_phase1."""
    from_year = from_year if from_year is not None else FROM_YEAR
    start = datetime(from_year, 1, 1, tzinfo=timezone.utc)
    epochs = [start + timedelta(days=spacing_hari * i) for i in range(n_sample)]

    log_step(f"Mengambil {n_sample} sampel epoch (jarak {spacing_hari} hari, mulai {from_year}-01-01) "
             f"untuk contoh transformasi wrapTo180...")
    moon_vals = get_ecliptic_lon(MOON_CMD, epochs)
    sun_vals = get_ecliptic_lon(SUN_CMD, epochs)
    log_ok(f"Contoh transformasi wrapTo180 selesai untuk {n_sample} epoch")

    rows = []
    for ep, m, s in zip(epochs, moon_vals, sun_vals):
        delta_raw = m - s
        f_t = wrap_to_180(delta_raw)
        wrapped = abs(delta_raw - f_t) > 1e-9
        rows.append({
            "Epoch UTC": _iso_z(ep),
            "Selisih Mentah delta_lambda (deg)": round(delta_raw, 6),
            "Hasil wrapTo180 f(t) (deg)": round(f_t, 6),
            "Keterangan": ("Nilai mentah melewati batas -180/180, dinormalisasi"
                            if wrapped else "Nilai sudah berada dalam rentang -180 s.d. 180"),
        })
    return pd.DataFrame(rows)


#### 11.3.1 Statistik Tahap Data Preparation & Modeling

In [ ]:
# Pastikan Bagian 11.1 (Fase 1) sudah dijalankan sebelum sel ini
df_statistik_dataprep_modeling = tampilkan_statistik_data_preparation_modeling()
df_statistik_dataprep_modeling


### 11.4 Contoh Transformasi wrapTo180 (Tabel 4.5)

In [ ]:
df_contoh_wrapto180 = contoh_transformasi_wrapto180(FROM_YEAR, n_sample=10, spacing_hari=15)
df_contoh_wrapto180


## 12. Pipeline KHGT Global (Grid Saksi Dunia)

Porting dari `src/lib/khgtPipeline.ts`. Ini adalah kolom **"Global"** pada Fase 2 — sesuai
Bab III skripsi, parameter geosentris/skenario global **hanya sebagai pembanding dan informasi
pendukung, bukan fokus utama penelitian** (fokus utama adalah Rule A/Rule B di Kota Bekasi,
Bagian 13).

Alur (identik dengan produksi):

1. **PKG1** — pindai grid titik daratan dunia (`ALL_POINTS` di bawah, ~90 titik), cari titik
   yang Matahari terbenamnya (a) sesudah konjungsi dan (b) sebelum tengah malam UTC hari
   konjungsi D, DAN altitude geosentris Bulan ≥ 5° serta elongasi geosentris ≥ 8° di sana.
   Jika ada yang lolos → 1 Ramadan = D+1 (saksi terbaik dipilih berdasarkan margin kelolosan
   tertinggi, dasi diputus oleh waktu terbenam paling awal).
2. **PKG2** — jika PKG1 tidak ada yang lolos: (a) periksa apakah konjungsi terjadi sebelum
   fajar astronomis (*nightEnd*, altitude Matahari ~-18°) di Wellington, Selandia Baru — jika
   tidak, langsung *istikmal* (1 Ramadan = D+2); (b) jika ya, pindai ulang khusus titik-titik
   Amerika untuk Matahari terbenam SESUDAH tengah malam UTC hari D.
3. Jika PKG1 dan PKG2 sama-sama tidak ada yang lolos → *istikmal* (1 Ramadan = D+2).

Catatan penyederhanaan yang disengaja (tidak memengaruhi tanggal hasil): pengayaan
observasi topocentris saksi (azimuth/altitude/iluminasi tampak dari lokasi saksi) dihilangkan
karena hanya bersifat tampilan tambahan pada aplikasi web dan tidak pernah dipakai untuk
menentukan `khgtStartCivilDate` / `pkgVariant` — satu-satunya nilai yang dipakai Fase 2.

In [ ]:
# Grid titik daratan dunia — disalin verbatim dari khgtPipeline.ts (SHORTLIST + GLOBAL_GRID)
SHORTLIST = [
    {"lat": 56.81, "lon": -158.86, "name": "Maklumat2026 witness", "americas": True},
    {"lat": 59.04, "lon": -158.52, "name": "Dillingham AK", "americas": True},
    {"lat": 58.80, "lon": -156.90, "name": "SW Alaska coast", "americas": True},
    {"lat": 64.50, "lon": -165.40, "name": "Nome AK", "americas": True},
    {"lat": 55.06, "lon": -162.32, "name": "Cold Bay AK", "americas": True},
    {"lat": 51.88, "lon": -176.65, "name": "Adak AK", "americas": True},
    {"lat": 61.22, "lon": -149.90, "name": "Anchorage AK", "americas": True},
    {"lat": 57.79, "lon": -152.41, "name": "Kodiak AK", "americas": True},
    {"lat": 55.34, "lon": -131.64, "name": "Ketchikan AK", "americas": True},
    {"lat": 47.61, "lon": -122.33, "name": "Seattle WA", "americas": True},
    {"lat": 37.77, "lon": -122.42, "name": "San Francisco CA", "americas": True},
    {"lat": 34.05, "lon": -118.24, "name": "Los Angeles CA", "americas": True},
    {"lat": 33.45, "lon": -112.07, "name": "Phoenix AZ", "americas": True},
    {"lat": 40.71, "lon": -74.01, "name": "New York NY", "americas": True},
    {"lat": 19.43, "lon": -99.13, "name": "Mexico City", "americas": True},
    {"lat": -12.05, "lon": -77.04, "name": "Lima Peru", "americas": True},
    {"lat": -34.60, "lon": -58.38, "name": "Buenos Aires", "americas": True},
    {"lat": -23.55, "lon": -46.63, "name": "Sao Paulo", "americas": True},
    {"lat": 4.71, "lon": -74.07, "name": "Bogota", "americas": True},
]

GLOBAL_GRID = [
    {"lat": 60, "lon": 25, "name": "N Europe", "americas": False},
    {"lat": 55, "lon": 10, "name": "N Germany", "americas": False},
    {"lat": 50, "lon": 0, "name": "UK/France", "americas": False},
    {"lat": 50, "lon": 15, "name": "Central Europe", "americas": False},
    {"lat": 50, "lon": 30, "name": "Ukraine", "americas": False},
    {"lat": 45, "lon": 10, "name": "N Italy", "americas": False},
    {"lat": 40, "lon": -4, "name": "Spain", "americas": False},
    {"lat": 40, "lon": 25, "name": "Greece", "americas": False},
    {"lat": 40, "lon": 40, "name": "Turkey E", "americas": False},
    {"lat": 55, "lon": 40, "name": "Russia W", "americas": False},
    {"lat": 55, "lon": 55, "name": "Russia Ural", "americas": False},
    {"lat": 55, "lon": 75, "name": "Russia W Siberia", "americas": False},
    {"lat": 55, "lon": 90, "name": "Russia C Siberia", "americas": False},
    {"lat": 55, "lon": 105, "name": "Russia E Siberia", "americas": False},
    {"lat": 55, "lon": 130, "name": "Russia Far East", "americas": False},
    {"lat": 50, "lon": 130, "name": "Russia Khabarovsk", "americas": False},
    {"lat": 35, "lon": -5, "name": "Morocco", "americas": False},
    {"lat": 30, "lon": 10, "name": "Libya", "americas": False},
    {"lat": 30, "lon": 30, "name": "Egypt", "americas": False},
    {"lat": 15, "lon": 30, "name": "Sudan", "americas": False},
    {"lat": 10, "lon": 40, "name": "Ethiopia", "americas": False},
    {"lat": 0, "lon": 30, "name": "E Africa", "americas": False},
    {"lat": -5, "lon": 35, "name": "Tanzania", "americas": False},
    {"lat": -15, "lon": 30, "name": "Zambia", "americas": False},
    {"lat": -25, "lon": 30, "name": "S Africa", "americas": False},
    {"lat": 5, "lon": 0, "name": "Ghana", "americas": False},
    {"lat": 10, "lon": 10, "name": "Nigeria", "americas": False},
    {"lat": -5, "lon": 15, "name": "Congo", "americas": False},
    {"lat": 35, "lon": 45, "name": "Iraq", "americas": False},
    {"lat": 25, "lon": 45, "name": "Saudi Arabia", "americas": False},
    {"lat": 25, "lon": 55, "name": "UAE", "americas": False},
    {"lat": 35, "lon": 55, "name": "Iran", "americas": False},
    {"lat": 35, "lon": 70, "name": "Afghanistan", "americas": False},
    {"lat": 40, "lon": 65, "name": "Uzbekistan", "americas": False},
    {"lat": 30, "lon": 70, "name": "Pakistan", "americas": False},
    {"lat": 25, "lon": 80, "name": "N India", "americas": False},
    {"lat": 20, "lon": 75, "name": "C India", "americas": False},
    {"lat": 10, "lon": 78, "name": "S India", "americas": False},
    {"lat": 28, "lon": 85, "name": "Nepal", "americas": False},
    {"lat": 24, "lon": 90, "name": "Bangladesh", "americas": False},
    {"lat": 40, "lon": 116, "name": "Beijing", "americas": False},
    {"lat": 30, "lon": 120, "name": "Shanghai", "americas": False},
    {"lat": 35, "lon": 135, "name": "Japan", "americas": False},
    {"lat": 37, "lon": 127, "name": "Korea", "americas": False},
    {"lat": 15, "lon": 100, "name": "Thailand", "americas": False},
    {"lat": 10, "lon": 106, "name": "Vietnam", "americas": False},
    {"lat": 5, "lon": 105, "name": "Malaysia/SG", "americas": False},
    {"lat": -6, "lon": 107, "name": "Jakarta", "americas": False},
    {"lat": -8, "lon": 115, "name": "Bali", "americas": False},
    {"lat": 15, "lon": 120, "name": "Philippines", "americas": False},
    {"lat": -25, "lon": 135, "name": "C Australia", "americas": False},
    {"lat": -35, "lon": 150, "name": "Sydney", "americas": False},
    {"lat": -37, "lon": 175, "name": "NZ North", "americas": False},
    {"lat": 60, "lon": -150, "name": "Alaska grid", "americas": True},
    {"lat": 55, "lon": -130, "name": "BC Canada", "americas": True},
    {"lat": 50, "lon": -110, "name": "Canada prairie", "americas": True},
    {"lat": 45, "lon": -90, "name": "US midwest", "americas": True},
    {"lat": 45, "lon": -70, "name": "US northeast", "americas": True},
    {"lat": 35, "lon": -100, "name": "US south", "americas": True},
    {"lat": 25, "lon": -100, "name": "Mexico N", "americas": True},
    {"lat": 10, "lon": -80, "name": "Panama", "americas": True},
    {"lat": -5, "lon": -60, "name": "Amazonia", "americas": True},
    {"lat": -15, "lon": -50, "name": "Brazil C", "americas": True},
    {"lat": -35, "lon": -65, "name": "Argentina", "americas": True},
    {"lat": 60, "lon": -160, "name": "W Alaska grid", "americas": True},
    {"lat": 65, "lon": -150, "name": "Interior Alaska", "americas": True},
    {"lat": 55, "lon": -160, "name": "Bristol Bay AK", "americas": True},
    {"lat": 57, "lon": -155, "name": "Katmai AK", "americas": True},
    {"lat": 53, "lon": -167, "name": "Unalaska AK", "americas": True},
]

ALL_POINTS = SHORTLIST + GLOBAL_GRID
print(f"Total titik grid saksi KHGT: {len(ALL_POINTS)}")


In [ ]:
# Zona waktu per titik grid: tz-lookup (Node) diganti timezonefinder (Python),
# lihat "Catatan Penting" di atas. Cache per-koordinat karena grid tetap sepanjang notebook.
from timezonefinder import TimezoneFinder

_tf = None
_tz_cache: Dict[Tuple[float, float], str] = {}


def get_timezone_for(lat: float, lon: float) -> str:
    global _tf
    key = (round(lat, 4), round(lon, 4))
    if key in _tz_cache:
        return _tz_cache[key]
    if _tf is None:
        _tf = TimezoneFinder()
    tz = _tf.timezone_at(lat=lat, lng=lon) or "UTC"
    _tz_cache[key] = tz
    return tz


def elongation_to_illumination_pct(elong_deg: float) -> float:
    """Port of khgtPipeline.ts:elongationToIlluminationPct."""
    elong_rad = elong_deg * math.pi / 180
    return (1 - math.cos(elong_rad)) / 2 * 100


In [ ]:
def scan_points(points: List[Dict], conjunction_utc: datetime, date_str: str,
                 after_midnight_d: bool) -> List[Dict]:
    """PKG1 (after_midnight_d=False, sunset before midnight D) / PKG2b
    (after_midnight_d=True, sunset after midnight D) witness-grid scan.
    Port of khgtPipeline.ts:scanPoints."""
    pkg_label = "PKG2 (Amerika, sunset setelah tengah malam D)" if after_midnight_d else "PKG1 (seluruh grid, sunset sebelum tengah malam D)"
    log_step(f"  {pkg_label}: memeriksa {len(points)} titik grid untuk konjungsi {date_str}...")
    midnight_d = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc) + timedelta(days=1)

    sunsets = []
    for pt in points:
        try:
            tz = get_timezone_for(pt["lat"], pt["lon"])
            r = get_sunset(date_str, pt["lat"], pt["lon"], tz)
            s_utc = r["sunsetUTC"]
            if s_utc <= conjunction_utc:
                sunsets.append(None)
                continue
            if not after_midnight_d:
                if s_utc >= midnight_d:
                    sunsets.append(None)
                    continue
            else:
                if s_utc < midnight_d:
                    sunsets.append(None)
                    continue
            sunsets.append({"point": pt, "sunsetUTC": s_utc, "sunsetLocal": r["sunsetLocal"], "tz": tz})
        except Exception:
            sunsets.append(None)

    valid = [s for s in sunsets if s]
    if not valid:
        log_ok(f"  {pkg_label}: 0/{len(points)} titik memenuhi syarat waktu terbenam — tidak ada yang lolos")
        return []

    valid_epochs = [s["sunsetUTC"] for s in valid]
    (moon_radec, sun_radec), radec_failures = _run_concurrent_tolerant([
        lambda: get_geocentric_apparent_radec(MOON_CMD, valid_epochs),
        lambda: get_geocentric_apparent_radec(SUN_CMD, valid_epochs),
    ])
    if radec_failures:
        log_warn(f"  {pkg_label}: gagal mengambil RA/Dec geosentris setelah {MAX_RETRIES}x percobaan "
                 "— titik-titik ini dilewati untuk sementara (jalankan ulang sel untuk mencoba lagi).")
        return []

    results = []
    for s, (ra_m, dec_m), (ra_s, dec_s) in zip(valid, moon_radec, sun_radec):
        geo_alt = geocentric_alt_deg(ra_m, dec_m, s["point"]["lat"], s["point"]["lon"], s["sunsetUTC"])
        geo_elong = geocentric_elong_deg(ra_m, dec_m, ra_s, dec_s)
        geo_illum = elongation_to_illumination_pct(geo_elong)
        check = check_khgt(geo_alt, geo_elong)
        results.append({
            "point": s["point"], "sunsetUTC": s["sunsetUTC"], "sunsetLocal": s["sunsetLocal"], "tz": s["tz"],
            "geoAltDeg": geo_alt, "geoElongDeg": geo_elong, "geoIlluminationPct": geo_illum,
            "pass": check["pass"], "altMargin": check["altMargin"], "elongMargin": check["elongMargin"],
        })
    n_pass = sum(1 for r in results if r["pass"])
    log_ok(f"  {pkg_label}: {len(valid)}/{len(points)} titik punya waktu terbenam valid, {n_pass} lolos ambang KHGT")
    return results


def pick_witness(candidates: List[Dict]) -> Optional[Dict]:
    """Pick the passing candidate with the highest min(altMargin, elongMargin);
    ties (within 0.0001) broken by earliest sunset. Port of khgtPipeline.ts:pickWitness."""
    passed = [c for c in candidates if c["pass"]]
    if not passed:
        return None

    def cmp(a, b):
        sa = min(a["altMargin"], a["elongMargin"])
        sb = min(b["altMargin"], b["elongMargin"])
        if abs(sa - sb) > 0.0001:
            return -1 if sb < sa else 1  # higher score first
        d = (a["sunsetUTC"] - b["sunsetUTC"]).total_seconds()
        return -1 if d < 0 else (1 if d > 0 else 0)

    passed_sorted = sorted(passed, key=cmp_to_key(cmp))
    return passed_sorted[0]


def next_day(date_str: str) -> str:
    return (datetime.strptime(date_str, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")


In [ ]:
def evaluate_khgt_witness_for_conjunction(conj_date: datetime, conj_iso: str, year: int) -> Dict:
    """PKG1/PKG2 core logic for one already-located conjunction.
    Port of khgtPipeline.ts:evaluateKHGTWitnessForConjunction (topocentric witness
    observation enrichment omitted — see Section 12 markdown for why this is safe)."""
    warnings = []
    D = conj_date.strftime("%Y-%m-%d")
    log_step(f"Evaluasi saksi KHGT global untuk konjungsi {D} (tahun target {year})")

    pkg1_results = scan_points(ALL_POINTS, conj_date, D, False)
    pkg1_witness = pick_witness(pkg1_results)
    if pkg1_witness:
        log_ok(f"PKG1 lolos — saksi: {pkg1_witness['point']['name']} -> KHGT mulai {next_day(D)}")
        return {
            "year": year, "khgtStartCivilDate": next_day(D), "conjunctionUTC": conj_iso,
            "pkgVariant": "PKG1", "witnessName": pkg1_witness["point"]["name"],
            "scanSummary": {"totalCandidates": len(pkg1_results),
                             "pkg1Passed": sum(1 for c in pkg1_results if c["pass"]), "pkg2Passed": 0},
            "warnings": warnings,
        }
    log_warn("PKG1 tidak ada saksi yang lolos — lanjut memeriksa syarat PKG2 (fajar Wellington NZ)")

    nz_fajr = get_nz_fajr_nightend_utc(conj_iso)
    conj_before_nz_fajr = conj_date < nz_fajr

    if not conj_before_nz_fajr:
        warnings.append("PKG2(a) failed: conjunction not before NZ fajar nightEnd")
        log_fail(f"PKG2(a) gagal — konjungsi tidak sebelum fajar Wellington ({nz_fajr.isoformat()}) -> istikmal, KHGT mulai {next_day(next_day(D))}")
        return {
            "year": year, "khgtStartCivilDate": next_day(next_day(D)), "conjunctionUTC": conj_iso,
            "pkgVariant": "NONE", "witnessName": None,
            "scanSummary": {"totalCandidates": len(pkg1_results), "pkg1Passed": 0, "pkg2Passed": 0},
            "warnings": warnings,
        }
    log_ok("PKG2(a) terpenuhi — konjungsi sebelum fajar Wellington, lanjut memindai grid Amerika (PKG2b)")

    americas_points = [p for p in ALL_POINTS if p["americas"]]
    pkg2_results = scan_points(americas_points, conj_date, D, True)
    pkg2_witness = pick_witness(pkg2_results)
    if pkg2_witness:
        log_ok(f"PKG2 lolos — saksi: {pkg2_witness['point']['name']} -> KHGT mulai {next_day(D)}")
        return {
            "year": year, "khgtStartCivilDate": next_day(D), "conjunctionUTC": conj_iso,
            "pkgVariant": "PKG2", "witnessName": pkg2_witness["point"]["name"],
            "scanSummary": {"totalCandidates": len(pkg1_results) + len(pkg2_results), "pkg1Passed": 0,
                             "pkg2Passed": sum(1 for c in pkg2_results if c["pass"])},
            "warnings": warnings,
        }

    warnings.append("Neither PKG1 nor PKG2 produced a passing witness")
    log_fail(f"PKG1 dan PKG2 sama-sama tidak ada saksi yang lolos -> istikmal, KHGT mulai {next_day(next_day(D))}")
    return {
        "year": year, "khgtStartCivilDate": next_day(next_day(D)), "conjunctionUTC": conj_iso,
        "pkgVariant": "NONE", "witnessName": None,
        "scanSummary": {"totalCandidates": len(pkg1_results) + len(pkg2_results), "pkg1Passed": 0, "pkg2Passed": 0},
        "warnings": warnings,
    }


def predict_khgt_for_syawal(year: int, ramadan_conj_utc: datetime) -> Dict:
    """Port of khgtPipeline.ts:predictKHGTForSyawal."""
    window_start = ramadan_conj_utc + timedelta(days=24)
    window_end = ramadan_conj_utc + timedelta(days=36)
    expected = ramadan_conj_utc + timedelta(days=29.5)

    all_conj = find_conjunctions_in_range(window_start, window_end)
    if not all_conj:
        raise RuntimeError(f"No Syawal conjunction found in window {window_start.isoformat()} to {window_end.isoformat()}")

    best = min(all_conj, key=lambda c: abs((c.t - expected).total_seconds()))
    return evaluate_khgt_witness_for_conjunction(best.t, best.iso, year)


def find_nearest_conjunction(center: datetime, radius_days: float) -> Optional[ConjSimple]:
    found = find_conjunctions_in_range(center - timedelta(days=radius_days), center + timedelta(days=radius_days))
    if not found:
        return None
    return min(found, key=lambda c: abs((c.t - center).total_seconds()))


HIJRI_YEAR_DAYS = 354.36667


def predict_khgt_full_for_gregorian_year(year: int) -> List[Dict]:
    """All Ramadan+Syawal KHGT pairs whose 1 Ramadan falls in the given Gregorian
    year (usually 1, occasionally 0 or 2 — see khgtPipeline.ts docstring for the
    double/skip-year boundary this walk-outward approach is built to survive).
    Port of khgtPipeline.ts:predictKHGTFullForGregorianYear."""
    year_str = str(year)
    log_step(f"[Global/KHGT] Mencari seluruh siklus konjungsi yang mungkin jatuh di tahun {year}...")
    plausible_min = datetime(year - 1, 12, 25, tzinfo=timezone.utc)
    plausible_max = datetime(year, 12, 31, 23, 59, 59, tzinfo=timezone.utc)
    in_band = lambda t: plausible_min <= t <= plausible_max

    seed = find_nearest_conjunction(estimate_ramadan_conj_date(year), 60)
    if not seed:
        raise RuntimeError(f"No conjunction found near Ramadan estimate for year {year}")
    log_ok(f"Konjungsi acuan (seed) ditemukan: {seed.t.strftime('%Y-%m-%d')}")

    candidates = [seed]
    cursor = seed
    for _ in range(2):
        nxt = find_nearest_conjunction(cursor.t + timedelta(days=HIJRI_YEAR_DAYS), 20)
        if not nxt:
            break
        candidates.append(nxt)
        cursor = nxt
        if not in_band(nxt.t):
            break

    cursor = seed
    for _ in range(2):
        prv = find_nearest_conjunction(cursor.t - timedelta(days=HIJRI_YEAR_DAYS), 20)
        if not prv:
            break
        candidates.append(prv)
        cursor = prv
        if not in_band(prv.t):
            break

    seen, results = set(), []
    for cand in candidates:
        if cand.iso in seen:
            continue
        seen.add(cand.iso)
        if not in_band(cand.t):
            continue
        ramadan = evaluate_khgt_witness_for_conjunction(cand.t, cand.iso, cand.t.year)
        if not ramadan["khgtStartCivilDate"].startswith(year_str + "-"):
            continue
        log_step(f"[Global/KHGT] Menghitung 1 Syawal untuk siklus Ramadan {ramadan['khgtStartCivilDate']}...")
        syawal = predict_khgt_for_syawal(ramadan["year"], dtparser.isoparse(ramadan["conjunctionUTC"]))
        log_ok(f"1 Syawal (KHGT) untuk siklus ini: {syawal['khgtStartCivilDate']}")
        results.append({"ramadan": ramadan, "syawal": syawal})

    results.sort(key=lambda r: r["ramadan"]["khgtStartCivilDate"])
    log_ok(f"[Global/KHGT] Tahun {year}: {len(results)} siklus 1 Ramadan ditemukan jatuh di tahun ini")
    return results


## 13. Pipeline Lokal Wujudul Hilal (Rule A/Rule B) — Evaluasi Utama Skripsi

Porting dari `src/lib/ramadanFromSyaban.ts`. Ini adalah kolom **"Local"** pada Fase 2 —
evaluasi utama skripsi di Kota Bekasi. Untuk setiap tahun jangkar (`year-1`, `year`, `year+1`,
guna menangani kasus lintas-tahun), sistem:

1. Menentukan jendela pencarian konjungsi dari `estimate_ramadan1(anchor_year)` (±20 hari,
   diperlebar bertahap jika gagal — lihat `predict_from_anchor`).
2. Menjalankan Newton-Raphson (`find_conjunction`, Bagian 8) untuk menemukan konjungsi tunggal.
3. Mengevaluasi Rule A/Rule B pada tanggal konjungsi D; jika belum terpenuhi, mencoba D+1, D+2,
   D+3 (istikmal).
4. Begitu Rule A & Rule B terpenuhi pada tanggal kandidat: **1 Ramadan = tanggal kandidat + 1
   hari**, dengan verifikasi konsistensi Sya'ban (konjungsi sebelumnya harus benar-benar terjadi
   sebelum konjungsi ini).
5. `attach_syawal` mengulang pola yang sama pada bulan sinodis berikutnya untuk 1 Syawal
   (Idul Fitri) — dipakai untuk melengkapi hasil, walau Fase 2 hanya membandingkan tanggal
   1 Ramadan.

`predict_from_anchor` **selalu memakai estimator inline `estimate_ramadan1`** (jalur jangkar
`anchors_syaban.json` dilewati) karena file jangkar tersebut tidak ada di proyek ini — identik
dengan perilaku produksi saat ini (lihat Bagian 10).

In [ ]:
def predict_from_anchor(anchor_year: int, lat: float, lon: float, tz: str) -> Optional[Dict]:
    """Run the Rule A/Rule B pipeline for a single anchor year, return the first
    Ramadan-start result found (may cross the Gregorian year boundary), or None
    if the criteria are never met for D..D+3. Port of
    ramadanFromSyaban.ts:predictFromAnchor (inline-estimator branch only — see
    Section 13 markdown for why)."""
    log_step(f"[Local/Bekasi] Tahun jangkar {anchor_year}: mencari konjungsi awal (estimasi {estimate_ramadan1(anchor_year).strftime('%Y-%m-%d')})...")
    est_ramadan = estimate_ramadan1(anchor_year)
    window_start = est_ramadan - timedelta(days=20)
    window_end = est_ramadan + timedelta(days=20)

    try:
        conj = find_conjunction(window_start, window_end)
    except Exception:
        log_warn(f"[Local/Bekasi] Tahun jangkar {anchor_year}: jendela awal gagal, memperlebar ±20 hari...")
        w1s, w1e = window_start - timedelta(days=20), window_end + timedelta(days=20)
        try:
            conj = find_conjunction(w1s, w1e)
        except Exception:
            log_warn(f"[Local/Bekasi] Tahun jangkar {anchor_year}: masih gagal, memperlebar ±60 hari...")
            w2s, w2e = window_start - timedelta(days=60), window_end + timedelta(days=60)
            conj = find_conjunction(w2s, w2e)  # allowed to raise — caller catches it

    if not conj["converged"]:
        log_fail(f"[Local/Bekasi] Tahun jangkar {anchor_year}: Newton-Raphson tidak konvergen")
        return None

    conj_utc = conj["conjunctionUTC"]
    zone = pytz.timezone(tz)
    conj_local = conj_utc.astimezone(zone)
    conj_date_str = conj_local.strftime("%Y-%m-%d")

    candidates_checked = []
    for offset in range(0, 4):
        candidate_date = (datetime.strptime(conj_date_str, "%Y-%m-%d") + timedelta(days=offset)).strftime("%Y-%m-%d")

        sunset = get_sunset(candidate_date, lat, lon, tz)
        moon_topo = get_topo_azel(MOON_CMD, [sunset["sunsetUTC"]], lat, lon)
        moon_alt = moon_topo[0][1]  # (az, el)

        wh = check_wujudul_hilal(conj_utc, sunset["sunsetUTC"], moon_alt, candidate_date)
        candidates_checked.append({"date": candidate_date, "result": wh})
        verdict = "terpenuhi" if wh["fulfilled"] else "belum terpenuhi"
        log_step(f"[Local/Bekasi]   D+{offset} ({candidate_date}): Rule A={wh['ruleA']}, Rule B={wh['ruleB']} -> {verdict}")

        if wh["fulfilled"]:
            # Syaban consistency: verify the previous conjunction really precedes this one
            try:
                approx_prev = conj_utc - timedelta(days=29.53)
                prev_conj = find_conjunction(approx_prev - timedelta(days=5), approx_prev + timedelta(days=5))
                if prev_conj["conjunctionUTC"] >= conj_utc:
                    log_warn(f"[Local/Bekasi]   D+{offset}: gagal verifikasi konsistensi Sya'ban, lanjut ke kandidat berikutnya")
                    continue
            except Exception:
                pass  # cannot verify — proceed anyway (matches production)

            sunset_plus_one = sunset["sunsetUTC"].astimezone(zone) + timedelta(seconds=1)
            ramadan1 = (datetime.strptime(candidate_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
            log_ok(f"[Local/Bekasi] Tahun jangkar {anchor_year}: 1 Ramadan = {ramadan1}")

            return {
                "ramadan1LocalDate": ramadan1,
                "ramadanStartLocalDateTime": sunset_plus_one.isoformat(),
                "conjunctionUTC": conj["conjunctionISO"], "conjunctionLocal": conj_local.isoformat(),
                "sunsetLocal": sunset["sunsetLocal"], "sunsetUTC": sunset["sunsetUTC"].isoformat(),
                "moonAltitudeAtSunsetDeg": round(moon_alt, 6),
                "ruleA": wh["ruleA"], "ruleB": wh["ruleB"], "isBorderline": wh["isBorderline"],
                "converged": conj["converged"], "totalIterations": conj["totalIterations"],
                "timezone": tz, "candidatesChecked": candidates_checked,
                # Syawal fields — filled in by attach_syawal()
                "syawal1LocalDate": None, "syawalStartLocalDateTime": None,
                "lastFastingLocalDate": None, "ramadanLengthDays": None,
            }
    log_fail(f"[Local/Bekasi] Tahun jangkar {anchor_year}: Rule A/Rule B tidak terpenuhi pada D..D+3")
    return None  # Rule A/Rule B not met for any of D..D+3


def attach_syawal(result: Dict, lat: float, lon: float, tz: str) -> None:
    """Compute 1 Syawal (Idul Fitri) for a given Ramadan result — same Rule A/B
    pipeline on the next synodic month. Mutates `result` in place.
    Port of ramadanFromSyaban.ts:attachSyawal."""
    log_step(f"[Local/Bekasi] Menghitung 1 Syawal untuk siklus Ramadan {result['ramadan1LocalDate']}...")
    ramadan_conj_utc = dtparser.isoparse(result["conjunctionUTC"])
    win_start = ramadan_conj_utc + timedelta(days=24)
    win_end = ramadan_conj_utc + timedelta(days=35)

    try:
        syawal_conj = find_conjunction(win_start, win_end)
    except Exception:
        log_fail("[Local/Bekasi] 1 Syawal: konjungsi tidak ditemukan pada jendela pencarian")
        return
    if not syawal_conj["converged"]:
        log_fail("[Local/Bekasi] 1 Syawal: Newton-Raphson tidak konvergen")
        return

    zone = pytz.timezone(tz)
    conj_local = syawal_conj["conjunctionUTC"].astimezone(zone)
    conj_date_str = conj_local.strftime("%Y-%m-%d")

    for offset in range(-1, 3):
        cand_date = (datetime.strptime(conj_date_str, "%Y-%m-%d") + timedelta(days=offset)).strftime("%Y-%m-%d")

        sunset = get_sunset(cand_date, lat, lon, tz)
        moon_topo = get_topo_azel(MOON_CMD, [sunset["sunsetUTC"]], lat, lon)
        moon_alt = moon_topo[0][1]

        wh = check_wujudul_hilal(syawal_conj["conjunctionUTC"], sunset["sunsetUTC"], moon_alt, cand_date)

        if wh["fulfilled"]:
            sunset_dt = sunset["sunsetUTC"].astimezone(zone)
            syawal1 = (datetime.strptime(cand_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")

            result["syawalStartLocalDateTime"] = (sunset_dt + timedelta(seconds=1)).isoformat()
            result["syawal1LocalDate"] = syawal1
            result["lastFastingLocalDate"] = cand_date
            r1 = datetime.strptime(result["ramadan1LocalDate"], "%Y-%m-%d")
            s1 = datetime.strptime(syawal1, "%Y-%m-%d")
            result["ramadanLengthDays"] = (s1 - r1).days
            log_ok(f"[Local/Bekasi] 1 Syawal = {syawal1} (lama Ramadan {result['ramadanLengthDays']} hari)")
            return
    log_fail("[Local/Bekasi] 1 Syawal: Rule A/Rule B tidak terpenuhi pada kandidat manapun")


def predict_ramadan_multi(year: int, lat: float, lon: float, tz: str) -> Dict:
    """Predict Ramadan start(s) whose 1 Ramadan falls in the given Gregorian
    year — 0, 1, or 2 results. Port of ramadanFromSyaban.ts:predictRamadanMulti."""
    log_step(f"[Local/Bekasi] Tahun {year}: mencoba tahun jangkar {year-1}, {year}, {year+1}...")
    collected = []
    for ay in (year - 1, year, year + 1):
        try:
            result = predict_from_anchor(ay, lat, lon, tz)
        except Exception as e:
            log_fail(f"[Local/Bekasi] Tahun jangkar {ay} gagal total: {e}")
            result = None
        if result:
            start_year = datetime.strptime(result["ramadan1LocalDate"], "%Y-%m-%d").year
            already = any(r["ramadan1LocalDate"] == result["ramadan1LocalDate"] for r in collected)
            if start_year == year and not already:
                attach_syawal(result, lat, lon, tz)
                collected.append(result)

    collected.sort(key=lambda r: r["ramadan1LocalDate"])
    if collected:
        log_ok(f"[Local/Bekasi] Tahun {year}: {len(collected)} hasil 1 Ramadan jatuh di tahun ini")
    else:
        log_fail(f"[Local/Bekasi] Tahun {year}: tidak ada hasil 1 Ramadan yang jatuh di tahun ini")
    return {"results": collected, "primary": collected[0] if collected else None, "year": year}


## 14. Data Historis Resmi Indonesia & Selisih Tanggal

Porting dari `src/lib/officialHistory/seedIndonesia.ts` (tabel tanggal 1 Ramadan resmi hasil
Sidang Isbat Kementerian Agama RI, 2010–2026 — setiap entri sudah melalui pengecekan silang
2+ sumber independen, lihat komentar sumber di file aslinya) dan
`src/lib/officialHistory/resolve.ts` (`diffCivilDays`).

In [ ]:
OFFICIAL_INDONESIA: Dict[int, Dict[str, Any]] = {
    2010: {"officialDate": "2010-08-11", "hijriYear": 1431},
    2011: {"officialDate": "2011-08-01", "hijriYear": 1432},
    2012: {"officialDate": "2012-07-21", "hijriYear": 1433},
    2013: {"officialDate": "2013-07-10", "hijriYear": 1434},
    2014: {"officialDate": "2014-06-29", "hijriYear": 1435},
    2015: {"officialDate": "2015-06-18", "hijriYear": 1436},
    2016: {"officialDate": "2016-06-06", "hijriYear": 1437},
    2017: {"officialDate": "2017-05-27", "hijriYear": 1438},
    2018: {"officialDate": "2018-05-17", "hijriYear": 1439},
    2019: {"officialDate": "2019-05-06", "hijriYear": 1440},
    2020: {"officialDate": "2020-04-24", "hijriYear": 1441},
    2021: {"officialDate": "2021-04-13", "hijriYear": 1442},
    2022: {"officialDate": "2022-04-03", "hijriYear": 1443},
    2023: {"officialDate": "2023-03-23", "hijriYear": 1444},
    2024: {"officialDate": "2024-03-12", "hijriYear": 1445},
    2025: {"officialDate": "2025-03-01", "hijriYear": 1446},
    2026: {"officialDate": "2026-02-19", "hijriYear": 1447},
}
OFFICIAL_AUTHORITY = "Government of Indonesia"
OFFICIAL_INSTITUTION = "Ministry of Religious Affairs (Kementerian Agama RI)"


def diff_civil_days(a: Optional[str], b: Optional[str]) -> Optional[int]:
    """Pure calendar-date subtraction (a - b), both 'YYYY-MM-DD'.
    Port of officialHistory/resolve.ts:diffCivilDays."""
    if not a or not b:
        return None
    ta = datetime.strptime(a, "%Y-%m-%d")
    tb = datetime.strptime(b, "%Y-%m-%d")
    return (ta - tb).days


## 15. FASE 2 — Evaluasi History Global dan Local

Porting dari `src/app/api/evaluate/route.ts`. Untuk setiap tahun dalam rentang, menghitung
tiga variabel independen (kegagalan salah satu tidak pernah menyembunyikan dua lainnya, persis
seperti desain produksi dengan `Promise.allSettled`):

- **A. `khgtDate`** — hasil `predict_khgt_full_for_gregorian_year` (Bagian 12), skenario global.
- **B. `localDate`** — hasil `predict_ramadan_multi` di Kota Bekasi (Bagian 13), evaluasi utama.
- **C. `officialDate`** — dari tabel historis resmi Indonesia (Bagian 14), pembanding terbatas.

Lalu menghitung selisih hari berpasangan: `khgtVsLocalDays`, `khgtVsOfficialDays`,
`localVsOfficialDays`.

In [ ]:
def run_phase2_evaluation(from_year: int, to_year: int, lat: float, lon: float, tz: str) -> pd.DataFrame:
    """Port of /api/evaluate route.ts. For each Gregorian year: global (KHGT),
    local (Rule A/Rule B at lat/lon), and official (Indonesia Sidang Isbat)
    1-Ramadan dates, plus their pairwise day differences."""
    items = []
    for year in range(from_year, to_year + 1):
        log_step(f"===== Tahun {year} ({year - from_year + 1}/{to_year - from_year + 1}) =====")

        try:
            khgt_full = predict_khgt_full_for_gregorian_year(year)
            khgt_rows = [{"khgtDate": f["ramadan"]["khgtStartCivilDate"], "witness": f["ramadan"]["witnessName"]}
                         for f in khgt_full]
            log_ok(f"Tahun {year} [Global/KHGT] berhasil — {len(khgt_rows)} hasil")
        except Exception as e:
            log_fail(f"Tahun {year} [Global/KHGT] gagal: {e}")
            khgt_rows = []

        try:
            local_multi = predict_ramadan_multi(year, lat, lon, tz)
            local_dates = sorted(r["ramadan1LocalDate"] for r in local_multi["results"]
                                  if r["ramadan1LocalDate"].startswith(f"{year}-"))
            log_ok(f"Tahun {year} [Local/Bekasi] berhasil — {len(local_dates)} hasil")
        except Exception as e:
            log_fail(f"Tahun {year} [Local/Bekasi] gagal: {e}")
            local_dates = []

        official = OFFICIAL_INDONESIA.get(year)
        official_date = official["officialDate"] if official else None
        if official_date:
            log_ok(f"Tahun {year} [Historis/Resmi] ditemukan — {official_date}")
        else:
            log_warn(f"Tahun {year} [Historis/Resmi] tidak tersedia di tabel Bagian 14")
        log_ok(f"Tahun {year} selesai diproses")

        n = max(len(khgt_rows), len(local_dates), 1)
        for i in range(n):
            khgt_date = khgt_rows[i]["khgtDate"] if i < len(khgt_rows) else None
            witness = khgt_rows[i]["witness"] if i < len(khgt_rows) else None
            local_date = local_dates[i] if i < len(local_dates) else None

            items.append({
                "year": year, "khgtDate": khgt_date, "witness": witness, "localDate": local_date,
                "officialDate": official_date,
                "officialAuthority": OFFICIAL_AUTHORITY if official_date else None,
                "officialInstitution": OFFICIAL_INSTITUTION if official_date else None,
                "khgtVsLocalDays": diff_civil_days(khgt_date, local_date),
                "khgtVsOfficialDays": diff_civil_days(khgt_date, official_date),
                "localVsOfficialDays": diff_civil_days(local_date, official_date),
            })
    return pd.DataFrame(items)


### 15.1 Jalankan Fase 2

> Fase 2 menjalankan pemindaian grid saksi KHGT (Bagian 12) **dan** pipeline lokal Wujudul
> Hilal (Bagian 13) untuk setiap tahun — beberapa menit untuk rentang penuh 2017–2026.
> Progres dicetak per tahun.

In [ ]:
log_step(f"===== FASE 2: Evaluasi History Global dan Local {FROM_YEAR}-{TO_YEAR} di Kota Bekasi "
         f"(lat={LAT}, lon={LON}, tz={TZ}) =====")
t0 = time.time()
df_phase2 = run_phase2_evaluation(FROM_YEAR, TO_YEAR, LAT, LON, TZ)
log_ok(f"FASE 2 selesai dalam {time.time() - t0:.1f} detik")
df_phase2


### 15.2 Ringkasan Kesesuaian (untuk kutipan Bab IV)

Rekap sederhana ala `evaluation.ts` (OK jika selisih 0 hari) untuk tiap pasangan sumber
tanggal — bukan validasi hukum, melainkan pola kesesuaian hasil komputasi terhadap data
historis, sesuai batasan yang dinyatakan dalam skripsi (Bagian 15 markdown di atas).

In [ ]:
total_with_official = int(df_phase2["officialDate"].notna().sum())
match_local_official = int((df_phase2["localVsOfficialDays"] == 0).sum())
match_khgt_official = int((df_phase2["khgtVsOfficialDays"] == 0).sum())
match_khgt_local = int((df_phase2["khgtVsLocalDays"] == 0).sum())

print(f"Tahun dengan data historis resmi tersedia : {total_with_official}")
print(f"Local (Bekasi) cocok persis dengan Resmi   : {match_local_official}/{total_with_official}")
print(f"Global (KHGT) cocok persis dengan Resmi     : {match_khgt_official}/{total_with_official}")
print(f"Global (KHGT) cocok persis dengan Local     : {match_khgt_local}/{len(df_phase2)}")

df_phase2[["year", "khgtDate", "localDate", "officialDate",
           "khgtVsLocalDays", "khgtVsOfficialDays", "localVsOfficialDays"]]


## 16. Ekspor Hasil (Excel untuk Lampiran Skripsi)

CSV sengaja TIDAK dipakai di sini: CSV adalah format teks polos tanpa metadata kolom, jadi
saat dibuka di Excel, hasilnya bergantung sepenuhnya pada pengaturan region Windows/Excel
pengguna (pemisah kolom `,` vs `;`) — di banyak Excel berlokal Indonesia/Eropa, CSV
ber-pemisah koma malah dianggap SATU kolom teks panjang per baris (persis yang terjadi di
screenshot: seluruh baris menumpuk di kolom A). CSV "netral" itu justru sumber masalahnya,
karena tidak ada yang menegaskan struktur kolomnya ke Excel.

Sel ini sebagai gantinya menulis **satu file Excel (`.xlsx`)** berisi 2 sheet (Fase 1 dan
Fase 2) memakai `openpyxl` — format `.xlsx` menyimpan batas kolom secara eksplisit di dalam
filenya sendiri, jadi selalu terbuka rapi di Excel apa pun pengaturan regionnya. Lebar kolom
juga disesuaikan otomatis (auto-fit) supaya langsung enak dibaca tanpa perlu dirapikan
manual.

In [ ]:
from openpyxl.utils import get_column_letter

def _autofit_excel_columns(worksheet, df: pd.DataFrame, max_width: int = 40) -> None:
    """Set each column's width to fit its longest value (header or data),
    capped at max_width so long text columns (e.g. candidateNote) don't blow
    up the sheet."""
    for i, col in enumerate(df.columns, start=1):
        values = df[col].astype(str)
        longest = max([len(str(col))] + [len(v) for v in values]) if len(df) else len(str(col))
        worksheet.column_dimensions[get_column_letter(i)].width = min(longest + 2, max_width)


log_step("Mengekspor hasil Fase 1 & Fase 2 ke satu file Excel (.xlsx)...")
EXCEL_FILENAME = "evaluasi_wujudul_hilal.xlsx"
with pd.ExcelWriter(EXCEL_FILENAME, engine="openpyxl") as writer:
    df_phase1_detail.to_excel(writer, sheet_name="Fase 1 - Konjungsi", index=False)
    _autofit_excel_columns(writer.sheets["Fase 1 - Konjungsi"], df_phase1_detail)

    df_phase2.to_excel(writer, sheet_name="Fase 2 - History Global Local", index=False)
    _autofit_excel_columns(writer.sheets["Fase 2 - History Global Local"], df_phase2)

log_ok(f"File Excel berhasil dibuat di penyimpanan sementara Colab (BUKAN folder Downloads komputer): {EXCEL_FILENAME}")
print("  - Sheet 1: Fase 1 - Konjungsi")
print("  - Sheet 2: Fase 2 - History Global Local")

try:
    from google.colab import files
    log_step("Memicu unduhan otomatis ke komputer kamu (izinkan pop-up bila diminta browser)...")
    files.download(EXCEL_FILENAME)
    log_ok("Unduhan dipicu — cek folder Downloads browser kamu.")
except ImportError:
    log_warn("Bukan runtime Google Colab — lewati unduhan otomatis. Ambil file lewat panel "
             "Files (ikon folder) di sidebar kiri: klik kanan file -> Download.")
except Exception as e:
    log_warn(f"Unduhan otomatis gagal ({e}). Ambil manual: buka panel Files (ikon folder) di "
             "sidebar kiri Colab, klik kanan file, lalu pilih Download.")


## 18. Lampiran: Reproduksi Detail Per-Langkah (untuk Bukti Sumber Data Bab IV)

Bagian ini **baru ditambahkan** khusus untuk menjawab catatan dosen pembimbing: *"angka-angka ini
didapat dari mana?"*. Semua fungsi di bagian ini **hanya memanggil** fungsi-fungsi yang sudah ada
di Bagian 5, 6, 8, dan 9 (`get_ecliptic_lon`, `wrap_to_180`, `get_sunset`, `get_topo_azel`,
`check_wujudul_hilal`, `scan_for_brackets`) — **tidak ada satu pun fungsi lama yang diubah**, jadi
seluruh Fase 1 dan Fase 2 di bagian atas tetap berjalan persis seperti sebelumnya.

Fungsi `demonstrasi_perhitungan_detail(...)` di bawah mencetak **setiap langkah substitusi angka**
persis seperti cara Bab IV skripsi menuliskan Persamaan (4.1)–(4.28) dan Tabel 4.8/4.11/4.12:
bujur ekliptika mentah dari Horizons, selisihnya (Persamaan 3.1), hasil `wrapTo180` (Persamaan
3.2), deteksi *sign change* (Persamaan 3.3), bracket & *initial guess* (Persamaan 3.4–3.5), lalu
**tiap iterasi Newton-Raphson** (turunan numerik Persamaan 3.7, koreksi waktu Persamaan 3.8,
update tebakan Persamaan 3.6) sampai konvergen, dan ditutup evaluasi Rule A/Rule B (Persamaan
3.9–3.10) untuk kandidat D sampai D+3 di Kota Bekasi.

**Cara pakai:** jalankan sel di Bagian 17.1 dengan koneksi Horizons aktif, lalu **screenshot
output teks + dua tabel yang muncul** (`df_iterasi_detail` dan `df_evaluasi_detail`) langsung dari
Colab untuk dilampirkan sebagai sumber data di skripsi. Tanggal default `"2026-02-17"` dipilih
supaya hasilnya bisa dibandingkan langsung dengan Tabel 4.8 yang sudah ada di skripsi — ganti
tanggalnya (mis. `"2017-05-25"`, `"2020-04-23"`, dst.) untuk mereproduksi baris Tabel 4.11/4.12
tahun lain.

In [ ]:
def demonstrasi_perhitungan_detail(target_date_str: str, lat: float = None, lon: float = None,
                                    tz: str = None, jendela_hari: int = 2):
    """Reproduksi manual Bab IV (Persamaan 4.1-4.28, Tabel 4.8/4.11/4.12) dengan angka Horizons
    sesungguhnya, dicetak per langkah agar bisa di-screenshot sebagai bukti sumber data.

    Fungsi ini BERDIRI SENDIRI: hanya memanggil scan_for_brackets, get_ecliptic_lon,
    wrap_to_180, get_sunset, get_topo_azel, dan check_wujudul_hilal yang sudah ada di
    Bagian 5/6/8/9 -- tidak memanggil atau mengubah try_nr_on_bracket / find_conjunction /
    run_phase1 / run_phase2_evaluation, jadi tidak berisiko terhadap Fase 1 & Fase 2 di atas.
    """
    lat = lat if lat is not None else LAT
    lon = lon if lon is not None else LON
    tz = tz if tz is not None else TZ

    print("=" * 78)
    print(f"CONTOH PERHITUNGAN DETAIL NEWTON-RAPHSON -- target tanggal {target_date_str}")
    print("Setara Persamaan (4.1)-(4.28) & Tabel 4.8 Bab IV skripsi")
    print("=" * 78)

    # ---- Langkah 1-2: pemindaian awal / bracketing (memakai scan_for_brackets Bagian 8) ----
    center = datetime.strptime(target_date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    window_start = center - timedelta(days=jendela_hari)
    window_end = center + timedelta(days=jendela_hari)
    print(f"\n[Langkah 1-2] Pemindaian awal (bracketing, langkah 6 jam) di sekitar {target_date_str}")
    brackets = scan_for_brackets(window_start, window_end)
    if not brackets:
        raise RuntimeError(f"Tidak ada bracket konjungsi ditemukan di sekitar {target_date_str}. "
                            f"Coba perbesar jendela_hari atau ganti tanggal.")
    bracket = brackets[0]
    t1, t2 = bracket["t1"], bracket["t2"]
    print(f"  Bracket ditemukan:")
    print(f"    t1 = {_iso_z(t1)}")
    print(f"    t2 = {_iso_z(t2)}")

    # ---- Langkah 3-4: bujur ekliptika & selisih -- Persamaan (3.1) ----
    print(f"\n[Langkah 3] Mengambil bujur ekliptika Bulan & Matahari dari NASA JPL Horizons")
    moon_t1, moon_t2 = get_ecliptic_lon(MOON_CMD, [t1, t2])
    sun_t1, sun_t2 = get_ecliptic_lon(SUN_CMD, [t1, t2])
    print(f"  lambda_Bulan(t1)    = {moon_t1:.6f} derajat")
    print(f"  lambda_Matahari(t1) = {sun_t1:.6f} derajat")
    print(f"  lambda_Bulan(t2)    = {moon_t2:.6f} derajat")
    print(f"  lambda_Matahari(t2) = {sun_t2:.6f} derajat")

    print(f"\n[Langkah 4] Menghitung selisih bujur ekliptika -- Persamaan (3.1): "
          f"delta_lambda(t) = lambda_Bulan(t) - lambda_Matahari(t)")
    delta_t1 = moon_t1 - sun_t1
    delta_t2 = moon_t2 - sun_t2
    print(f"  delta_lambda(t1) = {moon_t1:.6f} - {sun_t1:.6f} = {delta_t1:.6f} derajat")
    print(f"  delta_lambda(t2) = {moon_t2:.6f} - {sun_t2:.6f} = {delta_t2:.6f} derajat")

    # ---- Langkah 5: normalisasi -- Persamaan (3.2) ----
    print(f"\n[Langkah 5] Menormalisasi -- Persamaan (3.2): f(t) = wrapTo180(delta_lambda(t))")
    f_t1 = wrap_to_180(delta_t1)
    f_t2 = wrap_to_180(delta_t2)
    print(f"  f(t1) = wrapTo180({delta_t1:.6f}) = {f_t1:.6f} derajat")
    print(f"  f(t2) = wrapTo180({delta_t2:.6f}) = {f_t2:.6f} derajat")

    # ---- Langkah 6: sign change -- Persamaan (3.3) ----
    product = f_t1 * f_t2
    print(f"\n[Langkah 6] Deteksi perubahan tanda -- Persamaan (3.3): f(t1) x f(t2) < 0")
    print(f"  f(t1) x f(t2) = {f_t1:.6f} x {f_t2:.6f} = {product:.4f}")
    verdict63 = "TERDETEKSI PERUBAHAN TANDA (bracket valid)" if product < 0 else "TIDAK ADA PERUBAHAN TANDA"
    print(f"  {product:.4f} < 0 -> {verdict63}")

    # ---- Langkah 7-8: bracket & initial guess -- Persamaan (3.4)-(3.5) ----
    print(f"\n[Langkah 7] Bracket -- Persamaan (3.4): B = [t1, t2]")
    print(f"  B = [{_iso_z(t1)}, {_iso_z(t2)}]")

    t = t1 + (t2 - t1) / 2
    print(f"\n[Langkah 8] Initial guess -- Persamaan (3.5): t0 = (t1 + t2) / 2")
    print(f"  t0 = {_iso_z(t)}")

    # ---- Iterasi Newton-Raphson, dicetak tiap langkah ----
    print(f"\n{'-'*78}")
    print(f"ITERASI NEWTON-RAPHSON (delta={DELTA_S} detik, "
          f"toleransi |f|<{EPS_ANGLE} derajat ATAU |stepSec|<{EPS_TIME} detik, maks {MAX_ITER} iterasi)")
    print("-" * 78)

    iter_rows = []
    converged = False
    for i in range(MAX_ITER):
        t_minus = t - timedelta(seconds=DELTA_S)
        t_plus = t + timedelta(seconds=DELTA_S)

        moon_minus, moon_mid, moon_plus = get_ecliptic_lon(MOON_CMD, [t_minus, t, t_plus])
        sun_minus, sun_mid, sun_plus = get_ecliptic_lon(SUN_CMD, [t_minus, t, t_plus])

        f_minus = wrap_to_180(moon_minus - sun_minus)
        f_mid = wrap_to_180(moon_mid - sun_mid)
        f_plus = wrap_to_180(moon_plus - sun_plus)

        f_prime = (f_plus - f_minus) / (2 * DELTA_S)
        step_sec = 3600.0 if abs(f_prime) < 1e-15 else -(f_mid / f_prime)

        print(f"\n>> Iterasi {i + 1}")
        print(f"  [Langkah 9] f(t) dan turunan numerik -- Persamaan (3.7): "
              f"f'(tn) ~ [f(tn+delta) - f(tn-delta)] / (2*delta)")
        print(f"    f(t0 + {DELTA_S}s) = {f_plus:.6f} derajat")
        print(f"    f(t0 - {DELTA_S}s) = {f_minus:.6f} derajat")
        print(f"    f'(t0) = ({f_plus:.6f} - ({f_minus:.6f})) / {2 * DELTA_S} = {f_prime:.9f} derajat/detik")
        print(f"    f(t0) = {f_mid:.6f} derajat")
        print(f"  [Langkah 10] Koreksi waktu -- Persamaan (3.8): stepSec = -f(tn) / f'(tn)")
        print(f"    stepSec = -({f_mid:.6f} / {f_prime:.9f}) = {step_sec:.3f} detik")

        converged_by_angle = abs(f_mid) < EPS_ANGLE
        converged_by_time_and_angle = abs(step_sec) < EPS_TIME and abs(f_mid) < 0.01
        converged_this_step = converged_by_angle or converged_by_time_and_angle
        status = "Konvergen" if converged_this_step else "Belum konvergen"
        print(f"  Cek konvergensi: |f(t)|={abs(f_mid):.6f} (batas {EPS_ANGLE}), "
              f"|stepSec|={abs(step_sec):.3f} (batas {EPS_TIME}) -> {status}")

        iter_rows.append({
            "Iterasi": i + 1,
            "Waktu Tebakan UTC": _iso_z(t),
            "f(t) derajat": round(f_mid, 6),
            "f'(t) derajat/detik": round(f_prime, 9),
            "Koreksi Waktu detik": round(step_sec, 3),
            "Status": status,
        })

        if converged_this_step:
            converged = True
            break

        t_baru = t + timedelta(seconds=step_sec)
        print(f"  [Langkah 11] Update waktu tebakan -- Persamaan (3.6): t(n+1) = tn + stepSec")
        print(f"    t_baru = {_iso_z(t)} + ({step_sec:.3f} detik) = {_iso_z(t_baru)}")
        t = t_baru
        if t < window_start:
            t = window_start + timedelta(hours=1)
        if t > window_end:
            t = window_end - timedelta(hours=1)

    print(f"\n{'=' * 78}")
    if converged:
        print(f"[Langkah 13] KONVERGEN dalam {len(iter_rows)} iterasi.")
        print(f"  Waktu konjungsi final (UTC) = {_iso_z(t)}")
        zone = pytz.timezone(tz)
        t_local = t.astimezone(zone)
        print(f"  Setara waktu lokal ({tz}) = {t_local.strftime('%Y-%m-%d pukul %H:%M:%S.%f')[:-3]} {t_local.tzname()}")
    else:
        print(f"[PERINGATAN] Tidak konvergen dalam {MAX_ITER} iterasi.")
    print("=" * 78)

    df_iterasi = pd.DataFrame(iter_rows)
    conj_utc = t

    if not converged:
        return df_iterasi, pd.DataFrame()

    # ---- Langkah 14: evaluasi Rule A / Rule B, D..D+3 (gaya Tabel 4.11 & 4.12) ----
    print(f"\n{'-' * 78}")
    print("[Langkah 14] EVALUASI RULE A & RULE B DI KOTA BEKASI (D s.d. D+3)")
    print("-" * 78)
    conj_local_date = conj_utc.astimezone(pytz.timezone(tz)).strftime("%Y-%m-%d")

    eval_rows = []
    hasil_final = None
    for offset in range(0, 4):
        cand_date = (datetime.strptime(conj_local_date, "%Y-%m-%d") + timedelta(days=offset)).strftime("%Y-%m-%d")
        label = "D (hari konjungsi)" if offset == 0 else f"D+{offset}"
        sunset = get_sunset(cand_date, lat, lon, tz)
        topo = get_topo_azel(MOON_CMD, [sunset["sunsetUTC"]], lat, lon)[0]
        altitude = topo[1]
        wh = check_wujudul_hilal(conj_utc, sunset["sunsetUTC"], altitude, cand_date)

        print(f"\n  >> {label}: tanggal {cand_date}")
        print(f"     Waktu sunset (Bekasi) = {sunset['sunsetLocal']}")
        print(f"     Rule A -- Persamaan (3.9): t_konjungsi < t_sunset -> "
              f"{_iso_z(conj_utc)} < {_iso_z(sunset['sunsetUTC'])} -> {'Ya' if wh['ruleA'] else 'Tidak'}")
        print(f"     Altitude topocentric Bulan saat sunset = {altitude:.4f} derajat")
        print(f"     Rule B -- Persamaan (3.10): h_Bulan,topocentric(t_sunset) > 0 derajat -> "
              f"{altitude:.4f} > 0 -> {'Ya' if wh['ruleB'] else 'Tidak'}")
        status = "TERPENUHI" if wh["fulfilled"] else "Belum terpenuhi"
        print(f"     Status: {status}")

        eval_rows.append({
            "Tanggal": cand_date, "Label": label,
            "Rule A": "Ya" if wh["ruleA"] else "Tidak",
            "Altitude (deg)": round(altitude, 4),
            "Rule B": "Ya" if wh["ruleB"] else "Tidak",
            "Status": status,
        })

        if wh["fulfilled"] and hasil_final is None:
            ramadan1 = (datetime.strptime(cand_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
            hasil_final = ramadan1
            print(f"\n  >>> Rule A dan Rule B terpenuhi pada {cand_date} -> 1 Ramadan lokal = {ramadan1}")
            break

    df_evaluasi = pd.DataFrame(eval_rows)
    print(f"\n{'=' * 78}")
    if hasil_final:
        print(f"HASIL AKHIR: 1 Ramadan lokal (Kota Bekasi) = {hasil_final}")
    else:
        print("[PERINGATAN] Rule A/B belum terpenuhi sampai D+3 pada contoh ini.")
    print("=" * 78)

    return df_iterasi, df_evaluasi


### 18.1 Jalankan Demonstrasi (contoh: Ramadan 2026, sama dengan Tabel 4.8 skripsi)

In [ ]:
df_iterasi_detail, df_evaluasi_detail = demonstrasi_perhitungan_detail("2026-02-17", LAT, LON, TZ)


In [ ]:
# Tabel gaya Tabel 4.8 -- screenshot tabel ini
df_iterasi_detail


In [ ]:
# Tabel gaya Tabel 4.11/4.12 -- screenshot tabel ini
df_evaluasi_detail


### 18.2 Reproduksi Tahun Lain (opsional)

Ganti tanggal target untuk mereproduksi baris tahun lain di Tabel 4.11/4.12 (pakai tanggal
konjungsi dari Tabel 4.11, misalnya `"2017-05-25"`, `"2020-04-23"`, `"2024-03-10"`, dst).

In [ ]:
# Contoh: reproduksi tahun 2020 (kasus yang Rule A & B langsung terpenuhi di hari konjungsi)
# df_iterasi_2020, df_evaluasi_2020 = demonstrasi_perhitungan_detail("2020-04-23", LAT, LON, TZ)
# df_iterasi_2020
